# Eleições 2026 — tratamento e exportação com auditoria

Notebook revisado a partir de `V2_Analise_2026.ipynb`, do `.py` enviado e do catálogo de **28 arquivos / respectivas colunas**. Para Google Colab.

**O que foi validado nesta revisão:** leitura, vínculos, cálculos, exportação e regressões com dados sintéticos. Os CSVs reais, fotos e PDFs não estavam anexados; a auditoria desses dados acontecerá quando você executar este notebook no seu Drive. Nenhum teste transforma uma declaração do TSE em prova de veracidade material.

**Como executar**

1. Reinicie o ambiente do Colab, abra este notebook e confira os caminhos na configuração.
2. Mantenha os CSVs extraídos, sem edição manual de valores, diretamente em `Projeto_Eleicao2026`. As pastas de documentos podem conter subpastas.
3. Use **Ambiente de execução → Executar tudo**. Os testes rodam antes dos dados reais. Não precisa de chave Gemini.
4. Use apenas a pasta `validado_...` indicada ao final, com `EXPORTACAO_VALIDADA.json`. Se houver erro, abra `auditoria_bloqueio_...json` e corrija a fonte/recorte; não remova as validações.
5. A interface de busca aparece depois da exportação. Para apenas testar o código sem o Drive, selecione `somente_testes`.

**Atenção à integração do site:** a chave passa a incluir ano e eleição. Campos desconhecidos são `null`; o frontend deve mostrar **Não informado**, jamais converter `null` para zero. Leia a seção “Contrato de saída”. O código do site não foi enviado e não foi alterado nesta revisão.


## 1. Configuração

`BR` corresponde à abrangência/UF nacional de candidaturas como Presidência, conforme `SG_UF`; não significa automaticamente todas as UFs. O arquivo financeiro `BRASIL` contém todas as UFs e é filtrado por `SG_UF`. O padrão deste projeto continua **BR + MG**.

Para uma família com arquivo `BRASIL`, ele é utilizado sozinho; o arquivo `MG` redundante não é somado. Sem `BRASIL`, são utilizados os arquivos exatos de cada UF. Não existe busca parcial que possa escolher contas partidárias ou outro ano por engano.


In [ ]:
import subprocess
import sys

# Instalação das dependências de OCR necessárias no ambiente do Colab
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "PyMuPDF==1.26.6", "Pillow"
    ],
    check=True,
)

subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True,
)

subprocess.run(
    [
        "apt-get", "install", "-y", "-qq",
        "tesseract-ocr",
        "tesseract-ocr-por",
        "tesseract-ocr-eng",
    ],
    check=True,
)

print("Dependências de extração e OCR instaladas.")

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

BASE_DIR = "/content/drive/MyDrive/Projeto_Eleicao2026" #@param {type:"string"}
PASTA_SAIDA = "/content/drive/MyDrive/Projeto_Eleicao2026/exportacoes_auditadas" #@param {type:"string"}
MODO = "dados_reais" #@param ["dados_reais", "somente_testes"]
ANO_ELEICAO = 2026
UFS = ["BR", "MG"]
DATA_REFERENCIA_IDADE = datetime.now(ZoneInfo("America/Sao_Paulo")).date()
FORMATO_MONETARIO = "br"  # CSV bruto do TSE: 1.234,56. Não alterar sem confirmar a fonte.

# Opcional: nomes EXATOS de arquivos por família; use apenas um recorte coerente.
# Exemplo para ignorar BRASIL e usar arquivos por UF disponíveis:
# ESCOLHAS_ARQUIVOS = {"receitas": ["receitas_candidatos_2026_MG.csv", "receitas_candidatos_2026_BR.csv"]}
ESCOLHAS_ARQUIVOS = {}
ENCODINGS = {}  # Automático: UTF-8-SIG; Latin-1 apenas se falhar a decodificação.

# Para documento cujo nome não identifica ano+UF+SQ inequivocamente:
# só inclua após conferir o documento e sua candidatura; SHA-256 prende o vínculo aos bytes.
# {"caminhoRelativo":"Proposta_MG/nome.pdf", "tipo":"proposta",
#  "chave":"2026_CODIGO_ELEICAO_MG_SQ_CANDIDATO", "sha256":"HASH_COMPLETO_DO_ARQUIVO"}
VINCULOS_DOCUMENTOS = []
RESULTADO_DADOS_REAIS = None


In [ ]:
import importlib.util
import subprocess
import sys

# Colab já fornece pandas. Instale somente dependências ausentes, sem atualização geral.
for pacote in ["pandas", "pypdf"]:
    if importlib.util.find_spec(pacote) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",pacote])

if MODO == "dados_reais":
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path(BASE_DIR).is_dir():
        raise FileNotFoundError(f"Pasta não encontrada: {BASE_DIR}")
elif MODO != "somente_testes":
    raise ValueError("MODO inválido.")
print("Ambiente preparado. A exportação só ocorre depois dos testes e da auditoria.")


## 2. Mapeamento das relações

As chaves abaixo são regras de integração verificadas **em cada execução** por existência e cardinalidade. A lista de colunas não garante sozinha unicidade nem informa como combinar snapshots.

| Conjunto | Vínculo e regra | Valor / saída |
|---|---|---|
| `consulta_cand` | Ano + eleição + SQ; resultados separados por `NR_TURNO` | Biografia, partido, cargo, UF real, resultado de cada turno |
| `consulta_cand_complementar` | Ano + eleição + SQ → uma linha por candidatura | Situações, reeleição, substituição, declaração de bens, nascimento, FEFC, limite |
| `bem_candidato` | Ano + eleição + SQ + ordem do bem | `VR_BEM_CANDIDATO`; soma em centavos |
| `motivo_cassacao` | Ano + eleição + SQ; múltiplos motivos/processos permitidos | `DS_TP_MOTIVO`, `DS_MOTIVO`, `NR_PROCESSO` |
| `receitas_candidatos` | Ano + eleição + prestador + `SQ_RECEITA` | `VR_RECEITA`; candidato titular, nunca `SQ_CANDIDATO_DOADOR` |
| `despesas_contratadas_candidatos` | Ano + eleição + prestador + `SQ_DESPESA` | `VR_DESPESA_CONTRATADA`; candidato titular, nunca fornecedor |
| `despesas_pagas_candidatos` | Ano + eleição + prestador + despesa → contrato; parcela identifica o pagamento | **`VR_PAGTO_DESPESA`**; não existe `SQ_CANDIDATO` nessa fonte |
| `receitas_candidatos_doador_originario` | Ano + eleição + prestador + `SQ_RECEITA` → receita | Detalhamento; **não somar novamente na arrecadação** |

Receitas **e despesas contratadas** comprovam a relação entre `SQ_PRESTADOR_CONTAS` e `SQ_CANDIDATO`. Um candidato pode ter vários prestadores válidos. Um prestador da mesma eleição não pode apontar para vários candidatos. Cada parcela deve encontrar a despesa da mesma eleição/conta.

**Separação de conjuntos:** contas de órgãos partidários e contas anuais pertencem a outro universo contábil. Candidatos doadores/fornecedores nesses arquivos não são os titulares das contas. `denuncia_2026_*` não possui `SQ_CANDIDATO` no esquema enviado: município, cargo, partido e número de processo não autorizam atribuição automática a um candidato. Essas bases ficam fora dos totais/fichas de candidatos.

**Campos corrigidos:** resultados/turno vêm da principal; substituição, bens, protocolo, aceite e município de nascimento vêm da complementar. `SQ_SUBSTITUIDO` é exportado com nome neutro `sqSubstituido`, sem inventar a direção “substituído por”. `NM_UE` é unidade eleitoral, não necessariamente município. Gênero/cor FEFC ficam separados dos dados biográficos. Situações de urna, totalização, julgamento, cassação e diploma não se substituem entre si.

Fontes institucionais consultadas: [Candidatos 2026](https://dadosabertos.tse.jus.br/dataset/candidatos-2026) e [Prestação de contas eleitorais 2026](https://dadosabertos.tse.jus.br/dataset/prestacao-de-contas-eleitorais-2026). O catálogo de colunas abaixo reproduz o anexo; não representa inspeção das linhas dos CSVs.


In [ ]:
CATALOGO_COLUNAS = {'motivo_cassacao_2026_BR.csv': ['DT_GERACAO', 'HH_GERACAO', 'ANO_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'CD_ELEICAO', 'DS_ELEICAO', 'SG_UF', 'SG_UE', 'NM_UE', 'SQ_CANDIDATO', 'NR_PROCESSO', 'DS_TP_MOTIVO', 'DS_MOTIVO'], 'motivo_cassacao_2026_MG.csv': ['DT_GERACAO', 'HH_GERACAO', 'ANO_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'CD_ELEICAO', 'DS_ELEICAO', 'SG_UF', 'SG_UE', 'NM_UE', 'SQ_CANDIDATO', 'NR_PROCESSO', 'DS_TP_MOTIVO', 'DS_MOTIVO'], 'consulta_cand_complementar_2026_BR.csv': ['DT_GERACAO', 'HH_GERACAO', 'ANO_ELEICAO', 'CD_ELEICAO', 'SQ_CANDIDATO', 'CD_DETALHE_SITUACAO_CAND', 'DS_DETALHE_SITUACAO_CAND', 'CD_NACIONALIDADE', 'DS_NACIONALIDADE', 'CD_MUNICIPIO_NASCIMENTO', 'NM_MUNICIPIO_NASCIMENTO', 'NR_IDADE_DATA_POSSE', 'ST_QUILOMBOLA', 'CD_ETNIA_INDIGENA', 'DS_ETNIA_INDIGENA', 'VR_DESPESA_MAX_CAMPANHA', 'ST_REELEICAO', 'ST_DECLARAR_BENS', 'NR_PROTOCOLO_CANDIDATURA', 'NR_PROCESSO', 'CD_SITUACAO_CANDIDATO_PLEITO', 'DS_SITUACAO_CANDIDATO_PLEITO', 'CD_SITUACAO_CANDIDATO_URNA', 'DS_SITUACAO_CANDIDATO_URNA', 'ST_CANDIDATO_INSERIDO_URNA', 'NM_TIPO_DESTINACAO_VOTOS', 'CD_SITUACAO_CANDIDATO_TOT', 'DS_SITUACAO_CANDIDATO_TOT', 'ST_PREST_CONTAS', 'ST_SUBSTITUIDO', 'SQ_SUBSTITUIDO', 'SQ_ORDEM_SUPLENCIA', 'DT_ACEITE_CANDIDATURA', 'CD_SITUACAO_JULGAMENTO', 'DS_SITUACAO_JULGAMENTO', 'CD_SITUACAO_JULGAMENTO_PLEITO', 'DS_SITUACAO_JULGAMENTO_PLEITO', 'CD_SITUACAO_JULGAMENTO_URNA', 'DS_SITUACAO_JULGAMENTO_URNA', 'CD_SITUACAO_CASSACAO', 'DS_SITUACAO_CASSACAO', 'CD_SITUACAO_CASSACAO_MIDIA', 'DS_SITUACAO_CASSACAO_MIDIA', 'CD_SITUACAO_DIPLOMA', 'DS_SITUACAO_DIPLOMA', 'CD_GENERO_FEFC', 'DS_GENERO_FEFC', 'CD_COR_RACA_FEFC', 'DS_COR_RACA_FEFC'], 'consulta_cand_complementar_2026_MG.csv': ['DT_GERACAO', 'HH_GERACAO', 'ANO_ELEICAO', 'CD_ELEICAO', 'SQ_CANDIDATO', 'CD_DETALHE_SITUACAO_CAND', 'DS_DETALHE_SITUACAO_CAND', 'CD_NACIONALIDADE', 'DS_NACIONALIDADE', 'CD_MUNICIPIO_NASCIMENTO', 'NM_MUNICIPIO_NASCIMENTO', 'NR_IDADE_DATA_POSSE', 'ST_QUILOMBOLA', 'CD_ETNIA_INDIGENA', 'DS_ETNIA_INDIGENA', 'VR_DESPESA_MAX_CAMPANHA', 'ST_REELEICAO', 'ST_DECLARAR_BENS', 'NR_PROTOCOLO_CANDIDATURA', 'NR_PROCESSO', 'CD_SITUACAO_CANDIDATO_PLEITO', 'DS_SITUACAO_CANDIDATO_PLEITO', 'CD_SITUACAO_CANDIDATO_URNA', 'DS_SITUACAO_CANDIDATO_URNA', 'ST_CANDIDATO_INSERIDO_URNA', 'NM_TIPO_DESTINACAO_VOTOS', 'CD_SITUACAO_CANDIDATO_TOT', 'DS_SITUACAO_CANDIDATO_TOT', 'ST_PREST_CONTAS', 'ST_SUBSTITUIDO', 'SQ_SUBSTITUIDO', 'SQ_ORDEM_SUPLENCIA', 'DT_ACEITE_CANDIDATURA', 'CD_SITUACAO_JULGAMENTO', 'DS_SITUACAO_JULGAMENTO', 'CD_SITUACAO_JULGAMENTO_PLEITO', 'DS_SITUACAO_JULGAMENTO_PLEITO', 'CD_SITUACAO_JULGAMENTO_URNA', 'DS_SITUACAO_JULGAMENTO_URNA', 'CD_SITUACAO_CASSACAO', 'DS_SITUACAO_CASSACAO', 'CD_SITUACAO_CASSACAO_MIDIA', 'DS_SITUACAO_CASSACAO_MIDIA', 'CD_SITUACAO_DIPLOMA', 'DS_SITUACAO_DIPLOMA', 'CD_GENERO_FEFC', 'DS_GENERO_FEFC', 'CD_COR_RACA_FEFC', 'DS_COR_RACA_FEFC'], 'consulta_cand_2026_BR.csv': ['DT_GERACAO', 'HH_GERACAO', 'ANO_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'NR_TURNO', 'CD_ELEICAO', 'DS_ELEICAO', 'DT_ELEICAO', 'TP_ABRANGENCIA', 'SG_UF', 'SG_UE', 'NM_UE', 'CD_CARGO', 'DS_CARGO', 'SQ_CANDIDATO', 'NR_CANDIDATO', 'NM_CANDIDATO', 'NM_URNA_CANDIDATO', 'NM_SOCIAL_CANDIDATO', 'NR_CPF_CANDIDATO', 'DS_EMAIL', 'CD_SITUACAO_CANDIDATURA', 'DS_SITUACAO_CANDIDATURA', 'TP_AGREMIACAO', 'NR_PARTIDO', 'SG_PARTIDO', 'NM_PARTIDO', 'NR_FEDERACAO', 'NM_FEDERACAO', 'SG_FEDERACAO', 'DS_COMPOSICAO_FEDERACAO', 'SQ_COLIGACAO', 'NM_COLIGACAO', 'DS_COMPOSICAO_COLIGACAO', 'SG_UF_NASCIMENTO', 'DT_NASCIMENTO', 'NR_TITULO_ELEITORAL_CANDIDATO', 'CD_GENERO', 'DS_GENERO', 'CD_GRAU_INSTRUCAO', 'DS_GRAU_INSTRUCAO', 'CD_ESTADO_CIVIL', 'DS_ESTADO_CIVIL', 'CD_COR_RACA', 'DS_COR_RACA', 'CD_OCUPACAO', 'DS_OCUPACAO', 'CD_SIT_TOT_TURNO', 'DS_SIT_TOT_TURNO'], 'consulta_cand_2026_MG.csv': ['DT_GERACAO', 'HH_GERACAO', 'ANO_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'NR_TURNO', 'CD_ELEICAO', 'DS_ELEICAO', 'DT_ELEICAO', 'TP_ABRANGENCIA', 'SG_UF', 'SG_UE', 'NM_UE', 'CD_CARGO', 'DS_CARGO', 'SQ_CANDIDATO', 'NR_CANDIDATO', 'NM_CANDIDATO', 'NM_URNA_CANDIDATO', 'NM_SOCIAL_CANDIDATO', 'NR_CPF_CANDIDATO', 'DS_EMAIL', 'CD_SITUACAO_CANDIDATURA', 'DS_SITUACAO_CANDIDATURA', 'TP_AGREMIACAO', 'NR_PARTIDO', 'SG_PARTIDO', 'NM_PARTIDO', 'NR_FEDERACAO', 'NM_FEDERACAO', 'SG_FEDERACAO', 'DS_COMPOSICAO_FEDERACAO', 'SQ_COLIGACAO', 'NM_COLIGACAO', 'DS_COMPOSICAO_COLIGACAO', 'SG_UF_NASCIMENTO', 'DT_NASCIMENTO', 'NR_TITULO_ELEITORAL_CANDIDATO', 'CD_GENERO', 'DS_GENERO', 'CD_GRAU_INSTRUCAO', 'DS_GRAU_INSTRUCAO', 'CD_ESTADO_CIVIL', 'DS_ESTADO_CIVIL', 'CD_COR_RACA', 'DS_COR_RACA', 'CD_OCUPACAO', 'DS_OCUPACAO', 'CD_SIT_TOT_TURNO', 'DS_SIT_TOT_TURNO'], 'bem_candidato_2026_MG.csv': ['DT_GERACAO', 'HH_GERACAO', 'ANO_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'CD_ELEICAO', 'DS_ELEICAO', 'DT_ELEICAO', 'SG_UF', 'SG_UE', 'NM_UE', 'SQ_CANDIDATO', 'NR_ORDEM_BEM_CANDIDATO', 'CD_TIPO_BEM_CANDIDATO', 'DS_TIPO_BEM_CANDIDATO', 'DS_BEM_CANDIDATO', 'VR_BEM_CANDIDATO', 'DT_ULT_ATUAL_BEM_CANDIDATO', 'HH_ULT_ATUAL_BEM_CANDIDATO'], 'bem_candidato_2026_BR.csv': ['DT_GERACAO', 'HH_GERACAO', 'ANO_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'CD_ELEICAO', 'DS_ELEICAO', 'DT_ELEICAO', 'SG_UF', 'SG_UE', 'NM_UE', 'SQ_CANDIDATO', 'NR_ORDEM_BEM_CANDIDATO', 'CD_TIPO_BEM_CANDIDATO', 'DS_TIPO_BEM_CANDIDATO', 'DS_BEM_CANDIDATO', 'VR_BEM_CANDIDATO', 'DT_ULT_ATUAL_BEM_CANDIDATO', 'HH_ULT_ATUAL_BEM_CANDIDATO'], 'denuncia_2026_BRASIL.csv': ['DT_GERACAO', 'HH_GERACAO', 'AA_ELEICAO', 'DS_ELEICAO', 'SG_UF', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'DS_CARGO', 'TP_IRREGULARIDADE', 'DS_ORIGEM_DENUNCIA', 'DS_URL_PROCESSO_PJE', 'ST_PETICIONAMENTO', 'NR_PROCESSO_PJE'], 'denuncia_2026_MG.csv': ['DT_GERACAO', 'HH_GERACAO', 'AA_ELEICAO', 'DS_ELEICAO', 'SG_UF', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'DS_CARGO', 'TP_IRREGULARIDADE', 'DS_ORIGEM_DENUNCIA', 'DS_URL_PROCESSO_PJE', 'ST_PETICIONAMENTO', 'NR_PROCESSO_PJE'], 'receitas_orgaos_partidarios_doador_originario_2026_BRASIL.csv': ['DT_GERACAO', 'HH_GERACAO', 'AA_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'TP_PRESTACAO_CONTAS', 'DT_PRESTACAO_CONTAS', 'SQ_PRESTADOR_CONTAS', 'SG_UF', 'NR_CPF_CNPJ_DOADOR_ORIGINARIO', 'NM_DOADOR_ORIGINARIO', 'NM_DOADOR_ORIGINARIO_RFB', 'TP_DOADOR_ORIGINARIO', 'CD_CNAE_DOADOR_ORIGINARIO', 'DS_CNAE_DOADOR_ORIGINARIO', 'SQ_RECEITA', 'DT_RECEITA', 'DS_RECEITA', 'VR_RECEITA'], 'receitas_orgaos_partidarios_doador_originario_2026_MG.csv': ['DT_GERACAO', 'HH_GERACAO', 'AA_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'TP_PRESTACAO_CONTAS', 'DT_PRESTACAO_CONTAS', 'SQ_PRESTADOR_CONTAS', 'SG_UF', 'NR_CPF_CNPJ_DOADOR_ORIGINARIO', 'NM_DOADOR_ORIGINARIO', 'NM_DOADOR_ORIGINARIO_RFB', 'TP_DOADOR_ORIGINARIO', 'CD_CNAE_DOADOR_ORIGINARIO', 'DS_CNAE_DOADOR_ORIGINARIO', 'SQ_RECEITA', 'DT_RECEITA', 'DS_RECEITA', 'VR_RECEITA'], 'despesas_contratadas_orgaos_partidarios_2026_BRASIL.csv': ['DT_GERACAO', 'HH_GERACAO', 'AA_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'TP_PRESTACAO_CONTAS', 'DT_PRESTACAO_CONTAS', 'SQ_PRESTADOR_CONTAS', 'CD_ESFERA_PARTIDARIA', 'DS_ESFERA_PARTIDARIA', 'SG_UF', 'SG_UE', 'NM_UE', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'NR_CNPJ_PRESTADOR_CONTA', 'NR_PARTIDO', 'SG_PARTIDO', 'NM_PARTIDO', 'CD_TIPO_FORNECEDOR', 'DS_TIPO_FORNECEDOR', 'CD_CNAE_FORNECEDOR', 'DS_CNAE_FORNECEDOR', 'NR_CPF_CNPJ_FORNECEDOR', 'NM_FORNECEDOR', 'NM_FORNECEDOR_RFB', 'CD_ESFERA_PART_FORNECEDOR', 'DS_ESFERA_PART_FORNECEDOR', 'SG_UF_FORNECEDOR', 'CD_MUNICIPIO_FORNECEDOR', 'NM_MUNICIPIO_FORNECEDOR', 'SQ_CANDIDATO_FORNECEDOR', 'NR_CANDIDATO_FORNECEDOR', 'CD_CARGO_FORNECEDOR', 'DS_CARGO_FORNECEDOR', 'NR_PARTIDO_FORNECEDOR', 'SG_PARTIDO_FORNECEDOR', 'NM_PARTIDO_FORNECEDOR', 'DS_TIPO_DOCUMENTO', 'NR_DOCUMENTO', 'CD_ORIGEM_DESPESA', 'DS_ORIGEM_DESPESA', 'SQ_DESPESA', 'DT_DESPESA', 'DS_DESPESA', 'VR_DESPESA_CONTRATADA'], 'receitas_orgaos_partidarios_2026_MG.csv': ['DT_GERACAO', 'HH_GERACAO', 'AA_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'TP_PRESTACAO_CONTAS', 'DT_PRESTACAO_CONTAS', 'SQ_PRESTADOR_CONTAS', 'CD_ESFERA_PARTIDARIA', 'DS_ESFERA_PARTIDARIA', 'SG_UF', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'NR_CNPJ_PRESTADOR_CONTA', 'NR_PARTIDO', 'SG_PARTIDO', 'NM_PARTIDO', 'CD_FONTE_RECEITA', 'DS_FONTE_RECEITA', 'CD_ORIGEM_RECEITA', 'DS_ORIGEM_RECEITA', 'CD_NATUREZA_RECEITA', 'DS_NATUREZA_RECEITA', 'CD_ESPECIE_RECEITA', 'DS_ESPECIE_RECEITA', 'CD_CNAE_DOADOR', 'DS_CNAE_DOADOR', 'NR_CPF_CNPJ_DOADOR', 'NM_DOADOR', 'NM_DOADOR_RFB', 'CD_ESFERA_PARTIDARIA_DOADOR', 'DS_ESFERA_PARTIDARIA_DOADOR', 'SG_UF_DOADOR', 'CD_MUNICIPIO_DOADOR', 'NM_MUNICIPIO_DOADOR', 'SQ_CANDIDATO_DOADOR', 'NR_CANDIDATO_DOADOR', 'CD_CARGO_CANDIDATO_DOADOR', 'DS_CARGO_CANDIDATO_DOADOR', 'NR_PARTIDO_DOADOR', 'SG_PARTIDO_DOADOR', 'NM_PARTIDO_DOADOR', 'NR_RECIBO_DOACAO', 'NR_DOCUMENTO_DOACAO', 'SQ_RECEITA', 'DT_RECEITA', 'DS_RECEITA', 'VR_RECEITA'], 'receitas_orgaos_partidarios_2026_BRASIL.csv': ['DT_GERACAO', 'HH_GERACAO', 'AA_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'TP_PRESTACAO_CONTAS', 'DT_PRESTACAO_CONTAS', 'SQ_PRESTADOR_CONTAS', 'CD_ESFERA_PARTIDARIA', 'DS_ESFERA_PARTIDARIA', 'SG_UF', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'NR_CNPJ_PRESTADOR_CONTA', 'NR_PARTIDO', 'SG_PARTIDO', 'NM_PARTIDO', 'CD_FONTE_RECEITA', 'DS_FONTE_RECEITA', 'CD_ORIGEM_RECEITA', 'DS_ORIGEM_RECEITA', 'CD_NATUREZA_RECEITA', 'DS_NATUREZA_RECEITA', 'CD_ESPECIE_RECEITA', 'DS_ESPECIE_RECEITA', 'CD_CNAE_DOADOR', 'DS_CNAE_DOADOR', 'NR_CPF_CNPJ_DOADOR', 'NM_DOADOR', 'NM_DOADOR_RFB', 'CD_ESFERA_PARTIDARIA_DOADOR', 'DS_ESFERA_PARTIDARIA_DOADOR', 'SG_UF_DOADOR', 'CD_MUNICIPIO_DOADOR', 'NM_MUNICIPIO_DOADOR', 'SQ_CANDIDATO_DOADOR', 'NR_CANDIDATO_DOADOR', 'CD_CARGO_CANDIDATO_DOADOR', 'DS_CARGO_CANDIDATO_DOADOR', 'NR_PARTIDO_DOADOR', 'SG_PARTIDO_DOADOR', 'NM_PARTIDO_DOADOR', 'NR_RECIBO_DOACAO', 'NR_DOCUMENTO_DOACAO', 'SQ_RECEITA', 'DT_RECEITA', 'DS_RECEITA', 'VR_RECEITA'], 'despesas_contratadas_orgaos_partidarios_2026_MG.csv': ['DT_GERACAO', 'HH_GERACAO', 'AA_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'TP_PRESTACAO_CONTAS', 'DT_PRESTACAO_CONTAS', 'SQ_PRESTADOR_CONTAS', 'CD_ESFERA_PARTIDARIA', 'DS_ESFERA_PARTIDARIA', 'SG_UF', 'SG_UE', 'NM_UE', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'NR_CNPJ_PRESTADOR_CONTA', 'NR_PARTIDO', 'SG_PARTIDO', 'NM_PARTIDO', 'CD_TIPO_FORNECEDOR', 'DS_TIPO_FORNECEDOR', 'CD_CNAE_FORNECEDOR', 'DS_CNAE_FORNECEDOR', 'NR_CPF_CNPJ_FORNECEDOR', 'NM_FORNECEDOR', 'NM_FORNECEDOR_RFB', 'CD_ESFERA_PART_FORNECEDOR', 'DS_ESFERA_PART_FORNECEDOR', 'SG_UF_FORNECEDOR', 'CD_MUNICIPIO_FORNECEDOR', 'NM_MUNICIPIO_FORNECEDOR', 'SQ_CANDIDATO_FORNECEDOR', 'NR_CANDIDATO_FORNECEDOR', 'CD_CARGO_FORNECEDOR', 'DS_CARGO_FORNECEDOR', 'NR_PARTIDO_FORNECEDOR', 'SG_PARTIDO_FORNECEDOR', 'NM_PARTIDO_FORNECEDOR', 'DS_TIPO_DOCUMENTO', 'NR_DOCUMENTO', 'CD_ORIGEM_DESPESA', 'DS_ORIGEM_DESPESA', 'SQ_DESPESA', 'DT_DESPESA', 'DS_DESPESA', 'VR_DESPESA_CONTRATADA'], 'receitas_candidatos_2026_BRASIL.csv': ['DT_GERACAO', 'HH_GERACAO', 'AA_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'CD_ELEICAO', 'DS_ELEICAO', 'DT_ELEICAO', 'ST_TURNO', 'TP_PRESTACAO_CONTAS', 'DT_PRESTACAO_CONTAS', 'SQ_PRESTADOR_CONTAS', 'SG_UF', 'SG_UE', 'NM_UE', 'NR_CNPJ_PRESTADOR_CONTA', 'CD_CARGO', 'DS_CARGO', 'SQ_CANDIDATO', 'NR_CANDIDATO', 'NM_CANDIDATO', 'NR_CPF_CANDIDATO', 'NR_CPF_VICE_CANDIDATO', 'NR_PARTIDO', 'SG_PARTIDO', 'NM_PARTIDO', 'CD_FONTE_RECEITA', 'DS_FONTE_RECEITA', 'CD_ORIGEM_RECEITA', 'DS_ORIGEM_RECEITA', 'CD_NATUREZA_RECEITA', 'DS_NATUREZA_RECEITA', 'CD_ESPECIE_RECEITA', 'DS_ESPECIE_RECEITA', 'CD_CNAE_DOADOR', 'DS_CNAE_DOADOR', 'NR_CPF_CNPJ_DOADOR', 'NM_DOADOR', 'NM_DOADOR_RFB', 'CD_ESFERA_PARTIDARIA_DOADOR', 'DS_ESFERA_PARTIDARIA_DOADOR', 'SG_UF_DOADOR', 'CD_MUNICIPIO_DOADOR', 'NM_MUNICIPIO_DOADOR', 'SQ_CANDIDATO_DOADOR', 'NR_CANDIDATO_DOADOR', 'CD_CARGO_CANDIDATO_DOADOR', 'DS_CARGO_CANDIDATO_DOADOR', 'NR_PARTIDO_DOADOR', 'SG_PARTIDO_DOADOR', 'NM_PARTIDO_DOADOR', 'NR_RECIBO_DOACAO', 'NR_DOCUMENTO_DOACAO', 'SQ_RECEITA', 'DT_RECEITA', 'DS_RECEITA', 'VR_RECEITA', 'DS_NATUREZA_RECURSO_ESTIMAVEL', 'DS_GENERO', 'DS_COR_RACA'], 'despesas_contratadas_candidatos_2026_BRASIL.csv': ['DT_GERACAO', 'HH_GERACAO', 'AA_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'CD_ELEICAO', 'DS_ELEICAO', 'DT_ELEICAO', 'ST_TURNO', 'TP_PRESTACAO_CONTAS', 'DT_PRESTACAO_CONTAS', 'SQ_PRESTADOR_CONTAS', 'SG_UF', 'SG_UE', 'NM_UE', 'NR_CNPJ_PRESTADOR_CONTA', 'CD_CARGO', 'DS_CARGO', 'SQ_CANDIDATO', 'NR_CANDIDATO', 'NM_CANDIDATO', 'NR_CPF_CANDIDATO', 'NR_CPF_VICE_CANDIDATO', 'NR_PARTIDO', 'SG_PARTIDO', 'NM_PARTIDO', 'CD_TIPO_FORNECEDOR', 'DS_TIPO_FORNECEDOR', 'CD_CNAE_FORNECEDOR', 'DS_CNAE_FORNECEDOR', 'NR_CPF_CNPJ_FORNECEDOR', 'NM_FORNECEDOR', 'NM_FORNECEDOR_RFB', 'CD_ESFERA_PART_FORNECEDOR', 'DS_ESFERA_PART_FORNECEDOR', 'SG_UF_FORNECEDOR', 'CD_MUNICIPIO_FORNECEDOR', 'NM_MUNICIPIO_FORNECEDOR', 'SQ_CANDIDATO_FORNECEDOR', 'NR_CANDIDATO_FORNECEDOR', 'CD_CARGO_FORNECEDOR', 'DS_CARGO_FORNECEDOR', 'NR_PARTIDO_FORNECEDOR', 'SG_PARTIDO_FORNECEDOR', 'NM_PARTIDO_FORNECEDOR', 'DS_TIPO_DOCUMENTO', 'NR_DOCUMENTO', 'CD_ORIGEM_DESPESA', 'DS_ORIGEM_DESPESA', 'SQ_DESPESA', 'DT_DESPESA', 'DS_DESPESA', 'VR_DESPESA_CONTRATADA'], 'despesas_pagas_candidatos_2026_MG.csv': ['DT_GERACAO', 'HH_GERACAO', 'AA_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'CD_ELEICAO', 'DS_ELEICAO', 'DT_ELEICAO', 'ST_TURNO', 'TP_PRESTACAO_CONTAS', 'DT_PRESTACAO_CONTAS', 'SQ_PRESTADOR_CONTAS', 'SG_UF', 'DS_TIPO_DOCUMENTO', 'NR_DOCUMENTO', 'CD_FONTE_DESPESA', 'DS_FONTE_DESPESA', 'CD_ORIGEM_DESPESA', 'DS_ORIGEM_DESPESA', 'CD_NATUREZA_DESPESA', 'DS_NATUREZA_DESPESA', 'CD_ESPECIE_RECURSO', 'DS_ESPECIE_RECURSO', 'SQ_DESPESA', 'SQ_PARCELAMENTO_DESPESA', 'DT_PAGTO_DESPESA', 'DS_DESPESA', 'VR_PAGTO_DESPESA'], 'receitas_candidatos_doador_originario_2026_MG.csv': ['DT_GERACAO', 'HH_GERACAO', 'AA_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'CD_ELEICAO', 'DS_ELEICAO', 'DT_ELEICAO', 'ST_TURNO', 'TP_PRESTACAO_CONTAS', 'DT_PRESTACAO_CONTAS', 'SQ_PRESTADOR_CONTAS', 'SG_UF', 'NR_CPF_CNPJ_DOADOR_ORIGINARIO', 'NM_DOADOR_ORIGINARIO', 'NM_DOADOR_ORIGINARIO_RFB', 'TP_DOADOR_ORIGINARIO', 'CD_CNAE_DOADOR_ORIGINARIO', 'DS_CNAE_DOADOR_ORIGINARIO', 'SQ_RECEITA', 'DT_RECEITA', 'DS_RECEITA', 'VR_RECEITA'], 'despesas_pagas_candidatos_2026_BRASIL.csv': ['DT_GERACAO', 'HH_GERACAO', 'AA_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'CD_ELEICAO', 'DS_ELEICAO', 'DT_ELEICAO', 'ST_TURNO', 'TP_PRESTACAO_CONTAS', 'DT_PRESTACAO_CONTAS', 'SQ_PRESTADOR_CONTAS', 'SG_UF', 'DS_TIPO_DOCUMENTO', 'NR_DOCUMENTO', 'CD_FONTE_DESPESA', 'DS_FONTE_DESPESA', 'CD_ORIGEM_DESPESA', 'DS_ORIGEM_DESPESA', 'CD_NATUREZA_DESPESA', 'DS_NATUREZA_DESPESA', 'CD_ESPECIE_RECURSO', 'DS_ESPECIE_RECURSO', 'SQ_DESPESA', 'SQ_PARCELAMENTO_DESPESA', 'DT_PAGTO_DESPESA', 'DS_DESPESA', 'VR_PAGTO_DESPESA'], 'receitas_candidatos_doador_originario_2026_BRASIL.csv': ['DT_GERACAO', 'HH_GERACAO', 'AA_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'CD_ELEICAO', 'DS_ELEICAO', 'DT_ELEICAO', 'ST_TURNO', 'TP_PRESTACAO_CONTAS', 'DT_PRESTACAO_CONTAS', 'SQ_PRESTADOR_CONTAS', 'SG_UF', 'NR_CPF_CNPJ_DOADOR_ORIGINARIO', 'NM_DOADOR_ORIGINARIO', 'NM_DOADOR_ORIGINARIO_RFB', 'TP_DOADOR_ORIGINARIO', 'CD_CNAE_DOADOR_ORIGINARIO', 'DS_CNAE_DOADOR_ORIGINARIO', 'SQ_RECEITA', 'DT_RECEITA', 'DS_RECEITA', 'VR_RECEITA'], 'despesas_contratadas_candidatos_2026_MG.csv': ['DT_GERACAO', 'HH_GERACAO', 'AA_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'CD_ELEICAO', 'DS_ELEICAO', 'DT_ELEICAO', 'ST_TURNO', 'TP_PRESTACAO_CONTAS', 'DT_PRESTACAO_CONTAS', 'SQ_PRESTADOR_CONTAS', 'SG_UF', 'SG_UE', 'NM_UE', 'NR_CNPJ_PRESTADOR_CONTA', 'CD_CARGO', 'DS_CARGO', 'SQ_CANDIDATO', 'NR_CANDIDATO', 'NM_CANDIDATO', 'NR_CPF_CANDIDATO', 'NR_CPF_VICE_CANDIDATO', 'NR_PARTIDO', 'SG_PARTIDO', 'NM_PARTIDO', 'CD_TIPO_FORNECEDOR', 'DS_TIPO_FORNECEDOR', 'CD_CNAE_FORNECEDOR', 'DS_CNAE_FORNECEDOR', 'NR_CPF_CNPJ_FORNECEDOR', 'NM_FORNECEDOR', 'NM_FORNECEDOR_RFB', 'CD_ESFERA_PART_FORNECEDOR', 'DS_ESFERA_PART_FORNECEDOR', 'SG_UF_FORNECEDOR', 'CD_MUNICIPIO_FORNECEDOR', 'NM_MUNICIPIO_FORNECEDOR', 'SQ_CANDIDATO_FORNECEDOR', 'NR_CANDIDATO_FORNECEDOR', 'CD_CARGO_FORNECEDOR', 'DS_CARGO_FORNECEDOR', 'NR_PARTIDO_FORNECEDOR', 'SG_PARTIDO_FORNECEDOR', 'NM_PARTIDO_FORNECEDOR', 'DS_TIPO_DOCUMENTO', 'NR_DOCUMENTO', 'CD_ORIGEM_DESPESA', 'DS_ORIGEM_DESPESA', 'SQ_DESPESA', 'DT_DESPESA', 'DS_DESPESA', 'VR_DESPESA_CONTRATADA'], 'receitas_candidatos_2026_MG.csv': ['DT_GERACAO', 'HH_GERACAO', 'AA_ELEICAO', 'CD_TIPO_ELEICAO', 'NM_TIPO_ELEICAO', 'CD_ELEICAO', 'DS_ELEICAO', 'DT_ELEICAO', 'ST_TURNO', 'TP_PRESTACAO_CONTAS', 'DT_PRESTACAO_CONTAS', 'SQ_PRESTADOR_CONTAS', 'SG_UF', 'SG_UE', 'NM_UE', 'NR_CNPJ_PRESTADOR_CONTA', 'CD_CARGO', 'DS_CARGO', 'SQ_CANDIDATO', 'NR_CANDIDATO', 'NM_CANDIDATO', 'NR_CPF_CANDIDATO', 'NR_CPF_VICE_CANDIDATO', 'NR_PARTIDO', 'SG_PARTIDO', 'NM_PARTIDO', 'CD_FONTE_RECEITA', 'DS_FONTE_RECEITA', 'CD_ORIGEM_RECEITA', 'DS_ORIGEM_RECEITA', 'CD_NATUREZA_RECEITA', 'DS_NATUREZA_RECEITA', 'CD_ESPECIE_RECEITA', 'DS_ESPECIE_RECEITA', 'CD_CNAE_DOADOR', 'DS_CNAE_DOADOR', 'NR_CPF_CNPJ_DOADOR', 'NM_DOADOR', 'NM_DOADOR_RFB', 'CD_ESFERA_PARTIDARIA_DOADOR', 'DS_ESFERA_PARTIDARIA_DOADOR', 'SG_UF_DOADOR', 'CD_MUNICIPIO_DOADOR', 'NM_MUNICIPIO_DOADOR', 'SQ_CANDIDATO_DOADOR', 'NR_CANDIDATO_DOADOR', 'CD_CARGO_CANDIDATO_DOADOR', 'DS_CARGO_CANDIDATO_DOADOR', 'NR_PARTIDO_DOADOR', 'SG_PARTIDO_DOADOR', 'NM_PARTIDO_DOADOR', 'NR_RECIBO_DOACAO', 'NR_DOCUMENTO_DOACAO', 'SQ_RECEITA', 'DT_RECEITA', 'DS_RECEITA', 'VR_RECEITA', 'DS_NATUREZA_RECURSO_ESTIMAVEL', 'DS_GENERO', 'DS_COR_RACA'], 'despesa_anual_2026_MG.csv': ['DT_GERACAO', 'HH_GERACAO', 'AA_EXERCICIO', 'TP_DESPESA', 'CD_TP_ESFERA_PARTIDARIA', 'DS_TP_ESFERA_PARTIDARIA', 'SG_UF', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'NR_ZONA', 'NR_CNPJ_PRESTADOR_CONTA', 'SG_PARTIDO', 'NM_PARTIDO', 'CD_TP_DOCUMENTO', 'DS_TP_DOCUMENTO', 'NR_DOCUMENTO', 'AA_AIDF', 'NR_AIDF', 'CD_TP_FORNECEDOR', 'DS_TP_FORNECEDOR', 'NR_CPF_CNPJ_FORNECEDOR', 'NM_FORNECEDOR', 'DS_GASTO', 'DT_PAGAMENTO', 'VR_GASTO', 'VR_PAGAMENTO', 'VR_DOCUMENTO', 'CD_FONTE_DESPESA', 'DS_FONTE_DESPESA', 'SQ_DESPESA'], 'despesa_anual_2026_BRASIL.csv': ['DT_GERACAO', 'HH_GERACAO', 'AA_EXERCICIO', 'TP_DESPESA', 'CD_TP_ESFERA_PARTIDARIA', 'DS_TP_ESFERA_PARTIDARIA', 'SG_UF', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'NR_ZONA', 'NR_CNPJ_PRESTADOR_CONTA', 'SG_PARTIDO', 'NM_PARTIDO', 'CD_TP_DOCUMENTO', 'DS_TP_DOCUMENTO', 'NR_DOCUMENTO', 'AA_AIDF', 'NR_AIDF', 'CD_TP_FORNECEDOR', 'DS_TP_FORNECEDOR', 'NR_CPF_CNPJ_FORNECEDOR', 'NM_FORNECEDOR', 'DS_GASTO', 'DT_PAGAMENTO', 'VR_GASTO', 'VR_PAGAMENTO', 'VR_DOCUMENTO', 'CD_FONTE_DESPESA', 'DS_FONTE_DESPESA', 'SQ_DESPESA'], 'receita_anual_2026_MG.csv': ['DT_GERACAO', 'HH_GERACAO', 'CD_TP_ESFERA_PARTIDARIA', 'DS_TP_ESPERA_PARTIDARIA', 'SG_UF', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'NR_ZONA', 'NR_CNPJ_PRESTADOR_CONTA', 'SG_PARTIDO', 'NM_PARTIDO', 'CD_TP_ORIGEM_DOACAO', 'DS_TP_ORIGEM_DOACAO', 'NR_CPF_CNPJ_DOADOR', 'NM_DOADOR', 'CD_TP_ESFERA_PARTIDARIA_DOADOR', 'DS_TP_ESFERA_PARTIDARIA_DOADOR', 'SG_UF_DOADOR', 'CD_MUNICIPIO_DOADOR', 'NM_MUNICIPIO_DOADOR', 'NR_ZONA_DOADOR', 'SQ_CANDIDATO_DOADOR', 'NR_CANDIDATO_DOADOR', 'CD_CANDIDATO_CARGO_DOADOR', 'DS_CANDIDATO_CARGO_DOADOR', 'CD_TP_FONTE_RECURSO', 'DS_TP_FONTE_RECURSO', 'CD_TP_NATUREZA_RECURSO', 'DS_TP_NATUREZA_RECURSO', 'CD_TP_ESPECIE_RECURSO', 'DS_TP_ESPECIE_RECURSO', 'NR_RECIBO_DOACAO', 'NR_DOCUMENTO', 'DT_RECEITA', 'DS_RECEITA', 'VR_RECEITA'], 'receita_anual_2026_BRASIL.csv': ['DT_GERACAO', 'HH_GERACAO', 'CD_TP_ESFERA_PARTIDARIA', 'DS_TP_ESPERA_PARTIDARIA', 'SG_UF', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'NR_ZONA', 'NR_CNPJ_PRESTADOR_CONTA', 'SG_PARTIDO', 'NM_PARTIDO', 'CD_TP_ORIGEM_DOACAO', 'DS_TP_ORIGEM_DOACAO', 'NR_CPF_CNPJ_DOADOR', 'NM_DOADOR', 'CD_TP_ESFERA_PARTIDARIA_DOADOR', 'DS_TP_ESFERA_PARTIDARIA_DOADOR', 'SG_UF_DOADOR', 'CD_MUNICIPIO_DOADOR', 'NM_MUNICIPIO_DOADOR', 'NR_ZONA_DOADOR', 'SQ_CANDIDATO_DOADOR', 'NR_CANDIDATO_DOADOR', 'CD_CANDIDATO_CARGO_DOADOR', 'DS_CANDIDATO_CARGO_DOADOR', 'CD_TP_FONTE_RECURSO', 'DS_TP_FONTE_RECURSO', 'CD_TP_NATUREZA_RECURSO', 'DS_TP_NATUREZA_RECURSO', 'CD_TP_ESPECIE_RECURSO', 'DS_TP_ESPECIE_RECURSO', 'NR_RECIBO_DOACAO', 'NR_DOCUMENTO', 'DT_RECEITA', 'DS_RECEITA', 'VR_RECEITA']}

print(f"Catálogo de entrada: {len(CATALOGO_COLUNAS)} arquivos; cabeçalhos serão conferidos na leitura.")


## 3. Núcleo de integridade e leitura

Identificadores permanecem texto desde o CSV. A leitura confere todas as linhas, o número de campos, ano, cabeçalho e codificação. Cada fonte recebe SHA-256, contagem e data de geração declarada. `geradoEm` significa geração da exportação; não é apresentado como data da coleta do TSE.

Moeda é interpretada em formato brasileiro e somada como inteiro em centavos. Sentinela/ausência vira `None`; formato inválido bloqueia. Não há `except → 0`, `fillna(0)` financeiro ou tolerância de arredondamento escondendo divergências.


In [ ]:
# Núcleo auditável. Todas as funções abaixo também funcionam fora do Colab.
import csv
import hashlib
import json
import math
import re
import shutil
import unicodedata
import uuid
from collections import defaultdict
from datetime import date, datetime, timezone
from decimal import Decimal
from pathlib import Path
import pandas as pd

SCHEMA_VERSION = '3.0'
SENTINELAS = {'', '#NULO', '#NE', 'NAN', 'NONE', 'NULL', '-1', '-3'}
CK = ['ANO_ELEICAO', 'CD_ELEICAO', 'SQ_CANDIDATO']
PK = ['ANO_ELEICAO', 'CD_ELEICAO', 'SQ_PRESTADOR_CONTAS']
DK = PK + ['SQ_DESPESA']
RK = PK + ['SQ_RECEITA']
GERACAO = ['DT_GERACAO', 'HH_GERACAO']
VERSAO_CONTA = ['TP_PRESTACAO_CONTAS', 'DT_PRESTACAO_CONTAS', 'ST_TURNO']
FONTES_OFICIAIS = [
    'https://dadosabertos.tse.jus.br/dataset/candidatos-2026',
    'https://dadosabertos.tse.jus.br/dataset/prestacao-de-contas-eleitorais-2026',
]
FAMILIAS = {
    'candidatos': 'consulta_cand',
    'complementar': 'consulta_cand_complementar',
    'bens': 'bem_candidato',
    'cassacao': 'motivo_cassacao',
    'receitas': 'receitas_candidatos',
    'contratadas': 'despesas_contratadas_candidatos',
    'pagas': 'despesas_pagas_candidatos',
    'originarios': 'receitas_candidatos_doador_originario',
}
VALORES = {'bens': 'VR_BEM_CANDIDATO', 'receitas': 'VR_RECEITA',
           'contratadas': 'VR_DESPESA_CONTRATADA', 'pagas': 'VR_PAGTO_DESPESA',
           'originarios': 'VR_RECEITA', 'complementar': 'VR_DESPESA_MAX_CAMPANHA'}

class ErroIntegridade(RuntimeError):
    pass

class Auditoria:
    def __init__(self):
        self.eventos = []
        self.fontes = []
        self.escopo = {}
        self.reconciliacao = {}
    def registrar(self, nivel, codigo, mensagem, **contexto):
        self.eventos.append(dict(nivel=nivel, codigo=codigo, mensagem=mensagem, **contexto))
    @property
    def erros(self):
        return [x for x in self.eventos if x['nivel'] == 'ERRO']
    def exigir(self, condicao, codigo, mensagem, **contexto):
        if not condicao:
            self.registrar('ERRO', codigo, mensagem, **contexto)
    def barreira(self):
        if self.erros:
            codigos = ', '.join(dict.fromkeys(e['codigo'] for e in self.erros))
            raise ErroIntegridade(f'Exportação bloqueada: {len(self.erros)} ocorrência(s). {codigos}. Consulte a auditoria.')
    def relatorio(self):
        return dict(schemaVersion=SCHEMA_VERSION,
                    status='BLOQUEADO' if self.erros else 'VALIDADO_COM_AVISOS' if self.eventos else 'VALIDADO',
                    verificacao='Integridade técnica das fontes carregadas; não certifica a veracidade material das declarações.',
                    escopo=self.escopo, fontes=self.fontes, eventos=self.eventos,
                    reconciliacao=self.reconciliacao)


def limpar(v):
    if v is None or v is pd.NA or (isinstance(v, float) and math.isnan(v)):
        return None
    t = str(v).strip()
    return None if t.upper() in SENTINELAS else t


def id_texto(v):
    # Nunca passe por float/int: preserva dígitos e zeros iniciais.
    if not isinstance(v, str):
        raise ValueError('Identificador deve chegar como texto desde o CSV.')
    t = limpar(v)
    if t is None or not re.fullmatch(r'[0-9]+', t):
        raise ValueError('Identificador vazio, sentinela, decimal ou não numérico.')
    return t


def centavos(v, formato='br'):
    # Formato da FONTE, não adivinhação por célula. Nenhum arredondamento silencioso.
    t = limpar(v)
    if t is None or t in {'-1,00', '-3,00'}:
        return None
    if not isinstance(v, str):
        raise ValueError('Moeda deve ser lida como texto bruto.')
    if formato == 'br':
        if not re.fullmatch(r'(?:\d+|\d{1,3}(?:\.\d{3})+)(?:,\d{1,2})?', t):
            raise ValueError('Moeda fora do formato brasileiro esperado.')
        s = t.replace('.', '').replace(',', '.')
    elif formato == 'decimal_ponto':
        if not re.fullmatch(r'\d+(?:\.\d{1,2})?', t):
            raise ValueError('Moeda fora do formato decimal com ponto.')
        s = t
    else:
        raise ValueError('Formato monetário não suportado.')
    n = Decimal(s) * 100
    if n != n.to_integral_value():
        raise ValueError('Mais de duas casas decimais.')
    return int(n)


def reais(n):
    # Campo legado, apenas apresentação. Use *_centavos (string) para cálculos no site.
    return None if n is None else float(Decimal(n) / 100)


def moeda(n):
    if n is None:
        return 'Não informado'
    inteiro, cent = divmod(int(n), 100)
    return 'R$ ' + f'{inteiro:,}'.replace(',', '.') + f',{cent:02d}'


def hash_arquivo(p):
    h = hashlib.sha256()
    with Path(p).open('rb') as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b''):
            h.update(bloco)
    return h.hexdigest()


def refs(df):
    return [{'arquivo': r['__arquivo'], 'linhaFinalCSV': int(r['__linha'])}
            for _, r in df.iterrows()]


def colunas_dados(df):
    return [c for c in df.columns if not c.startswith('__') and c not in GERACAO]


def validar_ids(df, cols, aud, tabela):
    for c in cols:
        if c not in df:
            aud.registrar('ERRO', 'COLUNA_AUSENTE', f'{tabela}: falta {c}.')
            continue
        valores = []
        ruins = []
        for idx, v in df[c].items():
            try:
                valores.append(id_texto(v))
            except ValueError:
                valores.append(None)
                ruins.append(idx)
        df[c] = pd.Series(valores, index=df.index, dtype=object)
        if ruins:
            aud.registrar('ERRO', 'ID_INVALIDO', f'{tabela}.{c}: IDs inválidos.',
                          quantidade=len(ruins), exemplos=refs(df.loc[ruins].head(10)))


def ler_csv(p, esperado, ano, ufs, aud, encoding=None):
    p = Path(p)
    digest = hash_arquivo(p)
    def ler(enc):
        blocos, bloco, total, excluidos = [], [], 0, 0
        with p.open('r', encoding=enc, newline='') as f:
            reader = csv.reader(f, delimiter=';', strict=True)
            try:
                header = [x.strip().upper().lstrip('\ufeff') for x in next(reader)]
            except StopIteration:
                raise ErroIntegridade(f'CSV sem cabeçalho: {p.name}')
            if len(header) != len(set(header)):
                raise ErroIntegridade(f'Colunas duplicadas: {p.name}')
            if any(c.startswith('__') for c in header):
                raise ErroIntegridade(f'Prefixo interno reservado no cabeçalho: {p.name}')
            faltam = sorted(set(esperado) - set(header))
            if faltam:
                raise ErroIntegridade(f'{p.name}: colunas esperadas ausentes: {faltam}')
            iy = header.index('ANO_ELEICAO' if 'ANO_ELEICAO' in header else 'AA_ELEICAO')
            iu = header.index('SG_UF') if 'SG_UF' in header else None
            for row in reader:
                total += 1
                if len(row) != len(header):
                    raise ErroIntegridade(f'{p.name}, linha {reader.line_num}: {len(row)} campos; esperado {len(header)}.')
                if row[iy].strip() != str(ano):
                    raise ErroIntegridade(f'{p.name}, linha {reader.line_num}: ano diferente de {ano}.')
                if iu is not None and row[iu].strip() not in ufs:
                    excluidos += 1
                    continue
                bloco.append(row + [p.name, str(reader.line_num)])
                if len(bloco) >= 50000:
                    blocos.append(pd.DataFrame(bloco, columns=header + ['__arquivo','__linha']))
                    bloco = []
            if bloco or not blocos:
                blocos.append(pd.DataFrame(bloco, columns=header + ['__arquivo','__linha']))
        return pd.concat(blocos, ignore_index=True), total, excluidos
    encs = [encoding] if encoding else ['utf-8-sig', 'latin1']
    for enc in encs:
        try:
            df, total, excluidos = ler(enc)
            break
        except UnicodeDecodeError:
            if enc == encs[-1]:
                raise
    if hash_arquivo(p) != digest:
        raise ErroIntegridade(f'Fonte alterada durante a leitura: {p.name}')
    datas = df[GERACAO].drop_duplicates().to_dict('records')
    aud.fontes.append(dict(arquivo=p.name, sha256=digest, tamanhoBytes=p.stat().st_size,
                           encoding=enc, linhasLidas=total, linhasNoEscopo=len(df),
                           linhasForaDoEscopo=excluidos, geracaoDeclaradaTSE=datas,
                           colunas=list(df.columns[:-2])))
    for c in ['ANO_ELEICAO', 'AA_ELEICAO', 'CD_ELEICAO', 'SQ_CANDIDATO', 'SQ_PRESTADOR_CONTAS',
              'SQ_DESPESA', 'SQ_RECEITA', 'SQ_PARCELAMENTO_DESPESA', 'SG_UF']:
        if c in df:
            df[c] = df[c].str.strip()
    if 'AA_ELEICAO' in df:
        df = df.rename(columns={'AA_ELEICAO': 'ANO_ELEICAO'})
    return df


### Leitura das famílias e conciliação de duplicidades


In [ ]:
def selecionar_arquivos(base, familia, ano, ufs, aud, arquivos_explicitos=None):
    prefixo = FAMILIAS[familia]
    if arquivos_explicitos is not None:
        return [Path(base) / n for n in arquivos_explicitos]
    nacional = Path(base) / f'{prefixo}_{ano}_BRASIL.csv'
    if nacional.is_file():
        # BRASIL é cobertura nacional; BR é uma UF/abrangência e não seu sinônimo.
        ignorados = [f'{prefixo}_{ano}_{u}.csv' for u in ufs
                     if (Path(base) / f'{prefixo}_{ano}_{u}.csv').is_file()]
        if ignorados:
            aud.registrar('INFO', 'FONTE_REDUNDANTE_NAO_LIDA',
                          f'{familia}: escolhido BRASIL, filtrando SG_UF. Arquivos por UF não são somados.', arquivos=ignorados)
        return [nacional]
    return [Path(base) / f'{prefixo}_{ano}_{u}.csv' for u in ufs]


def carregar_fontes(base, ano, ufs, catalogo, aud, escolhas=None, encodings=None):
    tabelas, cobertura = {}, {}
    escolhas, encodings = escolhas or {}, encodings or {}
    aud.escopo = dict(ano=ano, ufs=list(ufs), criterio='SG_UF do registro; sem inferir UF pelo arquivo')
    for familia, prefixo in FAMILIAS.items():
        paths = selecionar_arquivos(base, familia, ano, ufs, aud, escolhas.get(familia))
        partes, cobertas = [], set()
        modelo = next(cols for nome, cols in catalogo.items() if nome.startswith(prefixo + '_2026_'))
        for p in paths:
            # Arquivos explícitos ainda precisam pertencer à família e ao ano.
            if not re.fullmatch(re.escape(prefixo) + '_' + str(ano) + r'_(?:BRASIL|[A-Z]{2})\.csv', p.name):
                aud.registrar('ERRO', 'ARQUIVO_INCOMPATIVEL', f'Nome não pertence à família/ano: {p.name}')
                continue
            if not p.is_file():
                aud.registrar('AVISO', 'FONTE_AUSENTE', f'{familia}: {p.name} não disponível.')
                continue
            try:
                df = ler_csv(p, modelo, ano, ufs, aud, encodings.get(p.name))
            except (ValueError, OSError, csv.Error, ErroIntegridade) as e:
                aud.registrar('ERRO', 'LEITURA_INVALIDA', str(e))
                continue
            declarada = p.stem.rsplit('_', 1)[1]
            if declarada == 'BRASIL':
                cobertas.update(ufs)
            else:
                cobertas.add(declarada)
                # Complementar não tem SG_UF: vínculo posterior recupera a UF real.
                if 'SG_UF' in df:
                    aud.exigir(df['SG_UF'].eq(declarada).all(), 'UF_ARQUIVO_DIVERGENTE',
                               f'{p.name}: contém outra UF no escopo. Não tratado automaticamente como BRASIL.')
            partes.append(df)
        tabelas[familia] = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame(
            columns=[('ANO_ELEICAO' if c == 'AA_ELEICAO' else c) for c in modelo] + ['__arquivo','__linha'])
        cobertura[familia] = cobertas
    aud.exigir(bool(len(tabelas['candidatos'])), 'SEM_CANDIDATOS', 'Nenhuma candidatura foi carregada.')
    aud.exigir(set(ufs) <= cobertura['candidatos'], 'ESCOPO_INCOMPLETO', 'Falta a base principal de uma UF solicitada.')
    conhecidos = {p['arquivo'] for p in aud.fontes}
    fora_pipeline = sorted(p.name for p in Path(base).glob('*.csv') if p.name not in conhecidos and any(token in p.name for token in ['orgaos_partidarios','denuncia_','receita_anual_','despesa_anual_']))
    if fora_pipeline:
        aud.registrar('INFO','BASES_SEPARADAS_DO_CANDIDATO','Arquivos partidários/anuais e denúncias não são atribuídos a candidatos por doador, fornecedor, partido ou município.', arquivos=fora_pipeline)
    aud.barreira()
    return tabelas, cobertura


def unicos(df, key, aud, tabela):
    """Conservador: repetições no mesmo CSV bloqueiam; cópias idênticas entre fontes são conciliadas."""
    if df.empty:
        return df.copy()
    validar_ids(df, key, aud, tabela)
    aud.barreira()
    dup = df.duplicated(key, keep=False)
    if not dup.any():
        return df.copy()
    manter = set(df.index[~dup])
    iguais = 0
    for _, grupo in df[dup].groupby(key, dropna=False, sort=False):
        if grupo['__arquivo'].duplicated().any():
            aud.registrar('ERRO', 'REGISTRO_REPETIDO_NA_FONTE',
                          f'{tabela}: chave repetida no mesmo arquivo. Revisar granularidade ou versões.', exemplos=refs(grupo.head(10)))
        elif len(grupo[colunas_dados(grupo)].drop_duplicates()) != 1:
            aud.registrar('ERRO', 'FONTES_CONFLITANTES',
                          f'{tabela}: mesma chave com conteúdo diferente; nenhuma versão escolhida silenciosamente.', exemplos=refs(grupo.head(10)))
        else:
            manter.add(grupo.index[0])
            iguais += len(grupo) - 1
    if iguais:
        aud.registrar('INFO', 'COPIAS_IDENTICAS_CONCILIADAS', f'{tabela}: {iguais} cópia(s) entre fontes não somada(s).')
    aud.barreira()
    return df.loc[sorted(manter)].reset_index(drop=True)


def join_seguro(left, right, key, aud, tabela, exigir_todos=True):
    n = len(left)
    out = left.merge(right, on=key, how='left', validate='many_to_one', indicator=True, suffixes=('', '__direita'))
    aud.exigir(len(out) == n, 'MULTIPLICACAO_JOIN', f'{tabela}: número de linhas alterado pelo vínculo.')
    soltos = out['_merge'].eq('left_only')
    if exigir_todos and soltos.any():
        aud.registrar('ERRO', 'VINCULO_AUSENTE', f'{tabela}: registros sem correspondente inequívoco.',
                      quantidade=int(soltos.sum()), exemplos=refs(out[soltos].head(10)))
    return out.drop(columns='_merge')


### Vínculos entre candidatura, conta, receita e parcela


In [ ]:
def validar_itens_contratados(df, aud):
    d = df.copy()
    validar_ids(d, DK, aud, 'contratadas')
    aud.barreira()

    fixos = [
        c for c in colunas_dados(d)
        if c not in {'DS_DESPESA', 'VR_DESPESA_CONTRATADA'}
    ]
    repetidas = d.duplicated(DK, keep=False)

    for chave, grupo in d.loc[repetidas].groupby(
        DK, sort=False, dropna=False
    ):
        contexto_unico = len(grupo[fixos].drop_duplicates()) == 1
        arquivo_unico = grupo['__arquivo'].nunique(dropna=False) == 1

        if not contexto_unico or not arquivo_unico:
            aud.registrar(
                'ERRO', 'ITENS_CONTRATADOS_CONFLITANTES',
                'A despesa reúne arquivos ou atributos incompatíveis.',
                chaveDespesa=list(chave),
                quantidade=len(grupo),
                exemplos=refs(grupo.head(10)),
            )
            continue

        linhas_identicas = int(
            grupo.duplicated(colunas_dados(grupo), keep=False).sum()
        )
        aud.registrar(
            'AVISO', 'DESPESA_COM_MULTIPLAS_LINHAS',
            'Linhas preservadas conforme a fonte, inclusive itens iguais; '
            'pagamentos serão vinculados à despesa sem multiplicação.',
            chaveDespesa=list(chave),
            quantidadeLinhas=len(grupo),
            linhasComConteudoIdentico=linhas_identicas,
            exemplos=refs(grupo.head(10)),
        )

    aud.barreira()
    return d.reset_index(drop=True)


def preparar_bases(t, aud):
    t = {k: v.copy() for k, v in t.items()}

    # Preserva os resultados de todos os turnos.
    t['candidatos'] = unicos(
        t['candidatos'], CK + ['NR_TURNO'], aud, 'candidatos'
    )
    cand = t['candidatos']
    estaveis = [
        c for c in colunas_dados(cand)
        if c not in [
            'NR_TURNO', 'CD_SIT_TOT_TURNO', 'DS_SIT_TOT_TURNO'
        ]
    ]
    base = cand[estaveis].drop_duplicates()

    aud.exigir(
        not base.duplicated(CK).any(),
        'CANDIDATURA_CONFLITANTE',
        'A mesma candidatura tem atributos divergentes entre turnos; '
        'preserve as versões para revisão.',
    )
    aud.barreira()

    base['__chave'] = (
        base['ANO_ELEICAO'] + '_' + base['CD_ELEICAO']
        + '_' + base['SG_UF'] + '_' + base['SQ_CANDIDATO']
    )
    aud.exigir(
        not base['__chave'].duplicated().any(),
        'CHAVE_JSON_DUPLICADA',
        'Colisão de chave de candidatura.',
    )

    identidade = base[
        CK + [
            'SG_UF', 'CD_CARGO', 'NR_CANDIDATO',
            'NR_CPF_CANDIDATO', '__chave'
        ]
    ]

    def interpretar_cpf(valor):
        # -4 significa CPF não disponível para comparação.
        if isinstance(valor, str) and valor.strip() == '-4':
            return None, 'ausente'

        texto = limpar(valor)
        if texto is None:
            return None, 'ausente'

        texto = re.sub(r'\s+', '', texto).upper().replace('X', '*')

        if re.fullmatch(r'[0-9*]{11}', texto):
            return texto, 'valido'

        if re.fullmatch(
            r'[0-9*]{3}\.[0-9*]{3}\.[0-9*]{3}-[0-9*]{2}', texto
        ):
            return texto.replace('.', '').replace('-', ''), 'valido'

        return None, 'formato_invalido'

    # Originários NÃO entram aqui: sua candidatura vem da ponte de contas.
    for nome in [
        'complementar', 'bens', 'cassacao', 'receitas', 'contratadas'
    ]:
        d = t[nome]
        validar_ids(d, CK, aud, nome)
        aud.barreira()

        if nome == 'complementar' and not d.empty:
            known = pd.MultiIndex.from_frame(base[CK])
            fora = ~pd.MultiIndex.from_frame(d[CK]).isin(known)
            nacional_fora = (
                fora & d['__arquivo'].str.endswith('_BRASIL.csv')
            )
            if nacional_fora.any():
                aud.registrar(
                    'INFO', 'COMPLEMENTAR_FORA_ESCOPO',
                    'Complementares nacionais fora do universo '
                    'de candidaturas excluídos.',
                    quantidade=int(nacional_fora.sum()),
                )
                d = d.loc[~nacional_fora].copy()

        d = join_seguro(d, identidade, CK, aud, nome)

        for c in [
            'SG_UF', 'CD_CARGO', 'NR_CANDIDATO', 'NR_CPF_CANDIDATO'
        ]:
            rc = c + '__direita'
            if rc not in d:
                continue

            if c == 'NR_CPF_CANDIDATO':
                problemas = []
                parciais = 0
                nao_comparaveis = 0

                for _, registro in d.iterrows():
                    cpf_origem, tipo_origem = interpretar_cpf(registro[c])
                    cpf_cadastro, tipo_cadastro = interpretar_cpf(registro[rc])
                    motivo = None
                    diferentes = []

                    if 'formato_invalido' in (tipo_origem, tipo_cadastro):
                        motivo = 'formato_invalido'
                    elif 'ausente' in (tipo_origem, tipo_cadastro):
                        nao_comparaveis += 1
                    else:
                        comparaveis = [
                            i for i in range(11)
                            if cpf_origem[i].isdigit()
                            and cpf_cadastro[i].isdigit()
                        ]
                        diferentes = [
                            i + 1 for i in comparaveis
                            if cpf_origem[i] != cpf_cadastro[i]
                        ]
                        if diferentes:
                            motivo = 'digitos_divergentes'
                        elif not comparaveis:
                            nao_comparaveis += 1
                        elif len(comparaveis) < 11:
                            parciais += 1

                    if motivo:
                        problemas.append({
                            'arquivo': registro['__arquivo'],
                            'linhaFinalCSV': int(registro['__linha']),
                            'SQ_CANDIDATO': registro['SQ_CANDIDATO'],
                            'CD_ELEICAO': registro['CD_ELEICAO'],
                            'motivo': motivo,
                            'posicoesDivergentes': diferentes,
                            'formatoOrigem': re.sub(
                                r'[0-9]', 'D', str(registro[c])
                            ),
                            'formatoCadastro': re.sub(
                                r'[0-9]', 'D', str(registro[rc])
                            ),
                        })

                if problemas:
                    aud.registrar(
                        'ERRO', 'IDENTIDADE_DIVERGENTE',
                        f'{nome}: CPF divergente ou formato não reconhecido.',
                        quantidade=len(problemas),
                        exemplos=problemas[:10],
                    )

                if parciais or nao_comparaveis:
                    aud.registrar(
                        'AVISO', 'CPF_COMPARACAO_LIMITADA',
                        f'{nome}: CPF não confirma identidade integral. '
                        'O vínculo continua dependendo de ano, eleição, '
                        'SQ_CANDIDATO e das demais validações.',
                        comparacoesParciais=parciais,
                        naoComparaveis=nao_comparaveis,
                    )
            else:
                origem = d[c].map(limpar)
                cadastro = d[rc].map(limpar)
                bad = (
                    origem.notna() & cadastro.notna()
                    & origem.ne(cadastro)
                )
                if bad.any():
                    aud.registrar(
                        'ERRO', 'IDENTIDADE_DIVERGENTE',
                        f'{nome}: {c} contradiz a base principal.',
                        quantidade=int(bad.sum()),
                        exemplos=refs(d.loc[bad].head(10)),
                    )

            d = d.drop(columns=rc)

        t[nome] = d

    aud.barreira()

    t['complementar'] = unicos(
        t['complementar'], CK, aud, 'complementar'
    )
    t['bens'] = unicos(
        t['bens'], CK + ['NR_ORDEM_BEM_CANDIDATO'], aud, 'bens'
    )

    # Motivos idênticos não representam novos processos.
    d = t['cassacao']
    payload = colunas_dados(d)
    repetidos = d.duplicated(payload, keep=False)
    if repetidos.any():
        aud.registrar(
            'AVISO', 'MOTIVO_REPETIDO',
            'Motivos exatamente repetidos foram consolidados; '
            'não representam novos processos.',
            quantidade=int(repetidos.sum()),
        )
        t['cassacao'] = d.drop_duplicates(payload).copy()

    # Constrói a ponte ANTES de separar registros sem lançamento.
    pontes = pd.concat([
        t[n][
            PK + ['SQ_CANDIDATO', '__chave', '__arquivo', '__linha']
        ]
        for n in ['receitas', 'contratadas']
    ], ignore_index=True)

    validar_ids(pontes, PK, aud, 'ponte_contas')
    aud.barreira()

    mapa = pontes[
        PK + ['SQ_CANDIDATO', '__chave']
    ].drop_duplicates()

    aud.exigir(
        not mapa.duplicated(PK).any(),
        'PRESTADOR_AMBIGUO',
        'Um prestador da mesma eleição aponta para mais de um candidato.',
    )
    aud.barreira()

    # Primeiro identifica a candidatura pela conta.
    for nome in ['pagas', 'originarios']:
        validar_ids(t[nome], PK, aud, nome)
        aud.barreira()

        t[nome] = join_seguro(t[nome], mapa, PK, aud, nome)
        ufs = base.set_index('__chave')['SG_UF']
        esperado = t[nome]['__chave'].map(ufs)
        bad = esperado.notna() & t[nome]['SG_UF'].ne(esperado)

        aud.exigir(
            not bad.any(),
            'UF_CONTA_DIVERGENTE',
            f'{nome}: UF da conta diverge da candidatura.',
        )

    aud.barreira()

    # Originários também participam da verificação de versão.
    versoes = pd.concat([
        t[n][CK + VERSAO_CONTA]
        for n in ['receitas', 'contratadas', 'pagas', 'originarios']
    ], ignore_index=True).drop_duplicates()

    for c in VERSAO_CONTA:
        aud.exigir(
            not versoes[c].map(limpar).isna().any(),
            'VERSAO_CONTA_AUSENTE',
            f'Falta {c} para distinguir prestações.',
        )

    versoes_comparacao = versoes.copy()
    versoes_comparacao['TP_PRESTACAO_CONTAS'] = (
        versoes_comparacao['TP_PRESTACAO_CONTAS'].map(
            lambda valor: (
                None if limpar(valor) is None
                else ' '.join(limpar(valor).split()).casefold()
            )
        )
    )
    versoes_comparacao = versoes_comparacao.drop_duplicates(
        CK + VERSAO_CONTA
    )
    conflitos = versoes_comparacao.loc[
        versoes_comparacao.duplicated(CK, keep=False)
    ]

    aud.exigir(
        conflitos.empty,
        'MULTIPLAS_PRESTACOES',
        'Há diferenças de tipo, data ou turno de prestação mesmo após '
        'normalizar maiúsculas/minúsculas e espaços.',
        candidaturasAfetadas=len(conflitos[CK].drop_duplicates()),
        exemplos=conflitos.head(15).to_dict('records'),
    )

    for v in versoes['DT_PRESTACAO_CONTAS'].unique():
        try:
            datetime.strptime(v, '%d/%m/%Y')
        except (ValueError, TypeError):
            aud.registrar(
                'ERRO', 'DATA_PRESTACAO_INVALIDA',
                'Data de prestação fora de DD/MM/AAAA.',
            )

    aud.barreira()

    # Separa marcadores sem receita.
    receitas = t['receitas']
    sem_lancamento = (
        receitas['SQ_RECEITA'].astype(str).str.strip().eq('-1')
        & receitas['VR_RECEITA'].map(limpar).isin(['0', '0,0', '0,00'])
        & receitas['DS_RECEITA'].map(limpar).isna()
        & receitas['DT_RECEITA'].map(limpar).isna()
    )
    t['receitas_sem_lancamento'] = receitas.loc[sem_lancamento].copy()

    if sem_lancamento.any():
        registros = []
        for _, r in t['receitas_sem_lancamento'].iterrows():
            registros.append({
                'arquivo': r['__arquivo'],
                'linhaFinalCSV': int(r['__linha']),
                **{
                    c: r[c]
                    for c in CK + ['SQ_PRESTADOR_CONTAS'] + VERSAO_CONTA
                },
                'SQ_RECEITA': r['SQ_RECEITA'],
                'VR_RECEITA': r['VR_RECEITA'],
                'DS_RECEITA': r['DS_RECEITA'],
                'DT_RECEITA': r['DT_RECEITA'],
            })
        aud.registrar(
            'AVISO', 'RECEITAS_SEM_LANCAMENTO_IDENTIFICADO',
            'Linhas com SQ_RECEITA=-1, valor zero, descrição e data '
            'ausentes foram separadas dos lançamentos. Não comprovam '
            'ausência de movimentação da campanha.',
            quantidade=len(registros),
            registros=registros,
        )

    t['receitas'] = receitas.loc[~sem_lancamento].copy()

    def validar_receitas_com_itens(df, aud):
        df = df.copy()
        validar_ids(df, RK, aud, 'receitas')
        aud.barreira()

        repetidas = df.duplicated(RK, keep=False)
        campos_item = ['DS_RECEITA', 'DS_NATUREZA_RECURSO_ESTIMAVEL']
        campos_variaveis = set(campos_item + ['VR_RECEITA'])

        for _, grupo in df.loc[repetidas].groupby(RK, sort=False):
            campos_fixos = [
                c for c in colunas_dados(grupo)
                if c not in campos_variaveis
            ]
            mesmo_contexto = (
                len(grupo[campos_fixos].drop_duplicates()) == 1
            )
            itens = grupo[campos_item].copy()
            for coluna in campos_item:
                itens[coluna] = itens[coluna].map(limpar)

            itens_descritos = itens.notna().all(axis=1).all()
            itens_distintos = not itens.duplicated().any()

            if not (
                mesmo_contexto and itens_descritos and itens_distintos
            ):
                aud.registrar(
                    'ERRO', 'RECEITA_REPETIDA_AMBIGUA',
                    'Mesmo SQ_RECEITA com contexto divergente, item sem '
                    'descrição ou item repetido. Nenhuma linha foi descartada.',
                    exemplos=refs(grupo.head(10)),
                )
            else:
                aud.registrar(
                    'AVISO', 'RECEITA_COM_ITENS',
                    'Mesmo SQ_RECEITA com itens descritos distintos e '
                    'demais campos iguais. Linhas preservadas separadamente.',
                    quantidadeItens=len(grupo),
                    exemplos=refs(grupo.head(10)),
                )

        aud.barreira()
        return df.reset_index(drop=True)

    t['receitas'] = validar_receitas_com_itens(t['receitas'], aud)

    # Separa marcadores sem despesa.
    contratadas = t['contratadas']
    sem_despesa = (
        contratadas['SQ_DESPESA'].astype(str).str.strip().eq('-1')
        & contratadas['VR_DESPESA_CONTRATADA'].map(limpar).isin(
            ['0', '0,0', '0,00']
        )
        & contratadas['DT_DESPESA'].map(limpar).isna()
        & contratadas['DS_DESPESA'].map(limpar).isna()
        & contratadas['DS_TIPO_DOCUMENTO'].map(limpar).isna()
        & contratadas['NR_DOCUMENTO'].map(
            lambda v: (
                limpar(v) is None
                or str(v).strip().upper() == '#NULO#'
            )
        )
    )
    t['contratadas_sem_lancamento'] = contratadas.loc[sem_despesa].copy()

    if sem_despesa.any():
        registros = []
        campos_preservados = (
            CK + ['SQ_PRESTADOR_CONTAS'] + VERSAO_CONTA
            + [
                'SQ_DESPESA', 'VR_DESPESA_CONTRATADA',
                'DT_DESPESA', 'DS_DESPESA',
                'DS_TIPO_DOCUMENTO', 'NR_DOCUMENTO',
            ]
        )
        for _, r in t['contratadas_sem_lancamento'].iterrows():
            registros.append({
                'arquivo': r['__arquivo'],
                'linhaFinalCSV': int(r['__linha']),
                **{c: r[c] for c in campos_preservados},
            })
        aud.registrar(
            'AVISO', 'CONTRATADAS_SEM_LANCAMENTO_IDENTIFICADO',
            'Linhas sem despesa identificada foram separadas dos contratos. '
            'Não comprovam ausência de movimentação da campanha.',
            quantidade=len(registros),
            registros=registros,
        )

    t['contratadas'] = validar_itens_contratados(
        contratadas.loc[~sem_despesa].copy(), aud
    )
    t['pagas'] = unicos(
        t['pagas'], DK + ['SQ_PARCELAMENTO_DESPESA'], aud, 'pagas'
    )

    # Um evento não pode migrar silenciosamente entre contas.
    for nome, seq in [
        ('receitas', 'SQ_RECEITA'),
        ('contratadas', 'SQ_DESPESA'),
        ('pagas', 'SQ_PARCELAMENTO_DESPESA'),
    ]:
        d = t[nome]
        chave_evento = (
            CK + (['SQ_DESPESA'] if nome == 'pagas' else []) + [seq]
        )
        contas_evento = d[
            chave_evento + ['SQ_PRESTADOR_CONTAS']
        ].drop_duplicates()
        aud.exigir(
            not contas_evento.duplicated(chave_evento).any(),
            'EVENTO_EM_MULTIPLAS_CONTAS',
            f'{nome}: mesmo evento em mais de uma conta da candidatura.',
        )

    aud.barreira()

    # Deduplica somente o mapa, nunca os itens monetários.
    contratos = (
        t['contratadas'][DK + ['SQ_CANDIDATO']]
        .drop_duplicates()
        .rename(columns={'SQ_CANDIDATO': '__candidato_contrato'})
    )
    aud.exigir(
        not contratos.duplicated(DK).any(),
        'DESPESA_CANDIDATO_AMBIGUO',
        'Uma mesma despesa aponta para mais de uma candidatura.',
    )
    aud.barreira()

    t['pagas'] = join_seguro(
        t['pagas'], contratos, DK, aud, 'pagamento_para_contrato'
    )
    aud.exigir(
        t['pagas']['SQ_CANDIDATO'].eq(
            t['pagas']['__candidato_contrato']
        ).all(),
        'PAGAMENTO_CANDIDATO_ERRADO',
        'A parcela não pertence ao candidato da despesa contratada.',
    )
    aud.barreira()

    # Originários já possuem candidatura validada pela ponte de contas.
    orig = t['originarios'].copy()

    def ausente_originario(valor):
        return (
            limpar(valor) is None
            or str(valor).strip().upper() == '#NULO#'
        )

    campos_sem_informacao = [
        'DT_RECEITA', 'DS_RECEITA',
        'NR_CPF_CNPJ_DOADOR_ORIGINARIO',
        'NM_DOADOR_ORIGINARIO', 'NM_DOADOR_ORIGINARIO_RFB',
        'TP_DOADOR_ORIGINARIO',
        'CD_CNAE_DOADOR_ORIGINARIO', 'DS_CNAE_DOADOR_ORIGINARIO',
    ]
    sem_lancamento = (
        orig['SQ_RECEITA'].astype(str).str.strip().eq('-1')
        & orig['VR_RECEITA'].astype(str).str.strip().isin(
            ['0', '0,0', '0,00']
        )
    )
    for coluna in campos_sem_informacao:
        sem_lancamento &= orig[coluna].map(ausente_originario)

    marcadores = orig.loc[sem_lancamento].copy()
    t['originarios_sem_lancamento'] = marcadores

    if not marcadores.empty:
        aud.registrar(
            'AVISO', 'ORIGINARIOS_SEM_LANCAMENTO',
            'Registros sem receita identificada e sem doador foram '
            'separados do detalhamento; vínculos de prestação preservados.',
            quantidade=len(marcadores),
            registros=marcadores[
                CK + ['SQ_PRESTADOR_CONTAS', 'SQ_RECEITA'] + VERSAO_CONTA
            ].to_dict('records'),
            referencias=refs(marcadores),
        )

    orig = orig.loc[~sem_lancamento].copy()
    validar_ids(orig, RK, aud, 'originarios')
    aud.barreira()

    recibos = (
        t['receitas'][RK + ['SQ_CANDIDATO']]
        .drop_duplicates()
        .rename(columns={'SQ_CANDIDATO': '__candidato_receita'})
    )
    aud.exigir(
        not recibos.duplicated(RK).any(),
        'RECEITA_CANDIDATO_AMBIGUO',
        'Uma receita aponta para mais de uma candidatura.',
    )
    aud.barreira()

    orig = join_seguro(
        orig, recibos, RK, aud, 'originario_para_receita'
    )
    aud.barreira()

    # Compara as duas relações; não preenche uma com a outra.
    aud.exigir(
        orig['SQ_CANDIDATO'].eq(orig['__candidato_receita']).all(),
        'ORIGINARIO_CANDIDATO_ERRADO',
        'Doador originário vinculado a outra candidatura.',
    )
    aud.barreira()

    repetidas = orig.duplicated(colunas_dados(orig), keep=False)
    orig['__originario_repetido'] = repetidas

    if repetidas.any():
        aud.registrar(
            'AVISO', 'ORIGINARIO_REPETIDO_PRESERVADO',
            'Detalhamento contém linhas indistinguíveis. Foram preservadas '
            'conforme a fonte, sem acréscimo ao total arrecadado. '
            'Não interpretar a quantidade de linhas como doadores únicos.',
            quantidadeLinhas=int(repetidas.sum()),
            exemplos=refs(orig.loc[repetidas].head(10)),
        )

    t['originarios'] = orig.reset_index(drop=True)
    aud.barreira()
    return t, base, mapa

### Validação das datas e dos valores


In [ ]:
def converter_limite_campanha(valor):
    """Conversão restrita à coluna VR_DESPESA_MAX_CAMPANHA."""
    texto = limpar(valor)

    if texto is None or texto in {
        '-1,00', '-3,00', '-1.00', '-3.00'
    }:
        return None

    if not isinstance(valor, str):
        raise ValueError('Limite deve ser texto bruto do CSV.')

    # Inteiro ou ponto decimal: 1234, 1234.5, 1234.56.
    if re.fullmatch(r'[0-9]+(?:\.[0-9]{1,2})?', texto):
        inteiro, _, fracao = texto.partition('.')
        numero_decimal = texto

    # Vírgula decimal, com ou sem separador de milhar.
    elif re.fullmatch(
        r'(?:[0-9]+|[0-9]{1,3}(?:\.[0-9]{3})+),[0-9]{1,2}',
        texto,
    ):
        inteiro, fracao = texto.replace('.', '').split(',')
        numero_decimal = texto.replace('.', '').replace(',', '.')

    else:
        # Não adivinha se "1.234" significa milhar ou três casas decimais.
        raise ValueError(
            'Limite inválido ou ambíguo; '
            'esperado 1234.56 ou 1.234,56.'
        )

    resultado = (
        int(inteiro) * 100
        + int(fracao.ljust(2, '0'))
    )

    # Conferência exata por outro método, sem float.
    if Decimal(numero_decimal) * 100 != resultado:
        raise ValueError('Falha na conferência exata do limite.')

    return resultado


def converter_valores(t, aud, formato='br'):
    for tabela, df in t.items():
        for col in [
            'DT_GERACAO',
            'DT_ELEICAO',
            'DT_RECEITA',
            'DT_DESPESA',
            'DT_PAGTO_DESPESA',
            'DT_ULT_ATUAL_BEM_CANDIDATO',
        ]:
            if col not in df:
                continue

            invalidos = []

            for idx, valor in df[col].items():
                valor = limpar(valor)
                if valor is None:
                    continue

                try:
                    datetime.strptime(valor, '%d/%m/%Y')
                except ValueError:
                    invalidos.append(idx)

            if invalidos:
                aud.registrar(
                    'ERRO',
                    'DATA_INVALIDA',
                    f'{tabela}.{col}: data inválida; '
                    'esperado DD/MM/AAAA.',
                    quantidade=len(invalidos),
                    exemplos=refs(df.loc[invalidos].head(10)),
                )

    for nome, coluna in VALORES.items():
        d = t[nome]
        valores = []
        invalidos = []

        for idx, valor in d[coluna].items():
            try:
                if nome == 'complementar':
                    convertido = converter_limite_campanha(valor)
                else:
                    # Mantém a regra existente para as outras fontes.
                    convertido = centavos(valor, formato)

                valores.append(convertido)

            except ValueError:
                valores.append(None)
                invalidos.append(idx)

        d['__centavos'] = pd.Series(
            valores, index=d.index, dtype=object
        )

        if invalidos:
            aud.registrar(
                'ERRO',
                'MOEDA_INVALIDA',
                f'{nome}.{coluna}: valores inválidos; '
                'nenhum convertido para zero.',
                quantidade=len(invalidos),
                exemplos=refs(d.loc[invalidos].head(10)),
            )

        ausentes = sum(v is None for v in valores)

        if ausentes:
            aud.registrar(
                'AVISO',
                'MOEDA_NAO_INFORMADA',
                f'{nome}: valores ausentes/sentinelas '
                'deixam o total correspondente nulo.',
                quantidade=ausentes,
            )

    aud.barreira()

    # Soma todos os itens de cada despesa, sem sobrescrever linhas.
    contratos = {}

    for chave, grupo in t['contratadas'].groupby(
        DK, sort=False, dropna=False
    ):
        valores = grupo['__centavos'].tolist()
        contratos[chave] = (
            sum(valores)
            if all(v is not None for v in valores)
            else None
        )

    # Cada parcela permanece contada apenas uma vez.
    for chave, grupo in t['pagas'].groupby(
        DK, sort=False, dropna=False
    ):
        if chave not in contratos:
            aud.registrar(
                'ERRO',
                'PAGAMENTO_SEM_CONTRATO',
                'Pagamento sem despesa contratada correspondente.',
                chaveDespesa=list(chave),
                exemplos=refs(grupo.head(10)),
            )
            continue

        contratado = contratos[chave]
        valores_pagos = grupo['__centavos'].tolist()

        if contratado is None or any(
            v is None for v in valores_pagos
        ):
            continue

        pago = sum(valores_pagos)

        aud.exigir(
            pago <= contratado,
            'PAGO_SUPERA_CONTRATO',
            'Parcelas superam a soma dos itens da despesa vinculada. '
            'Revisar versões/origem; não é conclusão de irregularidade.',
            chaveDespesa=list(chave),
            contratadoCentavos=str(contratado),
            pagoCentavos=str(pago),
        )

    aud.barreira()

In [ ]:
def converter_limite_campanha(valor):
    """Conversão restrita à coluna VR_DESPESA_MAX_CAMPANHA."""
    texto = limpar(valor)

    # Sentinelas representam informação indisponível, nunca zero.
    if texto is None or texto in {
        '-4', '-1,00', '-3,00', '-1.00', '-3.00'
    }:
        return None

    if not isinstance(valor, str):
        raise ValueError('Limite deve ser texto bruto do CSV.')

    if re.fullmatch(r'[0-9]+(?:\.[0-9]{1,2})?', texto):
        inteiro, _, fracao = texto.partition('.')
        numero_decimal = texto

    elif re.fullmatch(
        r'(?:[0-9]+|[0-9]{1,3}(?:\.[0-9]{3})+),[0-9]{1,2}',
        texto,
    ):
        inteiro, fracao = texto.replace('.', '').split(',')
        numero_decimal = texto.replace('.', '').replace(',', '.')

    else:
        raise ValueError(
            'Limite inválido ou ambíguo; '
            'esperado 1234.56 ou 1.234,56.'
        )

    resultado = int(inteiro) * 100 + int(fracao.ljust(2, '0'))

    if Decimal(numero_decimal) * 100 != resultado:
        raise ValueError('Falha na conferência exata do limite.')

    return resultado

### Projeção pública: biografia, patrimônio, finanças e situação jurídica


In [ ]:
MAPA_PRINCIPAL = {
    'numeroUrna': 'NR_CANDIDATO',
    'nomeUrna': 'NM_URNA_CANDIDATO',
    'nomeCompleto': 'NM_CANDIDATO',
    'nomeSocial': 'NM_SOCIAL_CANDIDATO',
    'cargo': 'DS_CARGO',
    'codigoCargo': 'CD_CARGO',
    'unidadeEleitoral': 'NM_UE',
    'codigoUnidadeEleitoral': 'SG_UE',
    'abrangencia': 'TP_ABRANGENCIA',
    'partido': 'SG_PARTIDO',
    'nomePartido': 'NM_PARTIDO',
    'coligacao': 'NM_COLIGACAO',
    'genero': 'DS_GENERO',
    'corRaca': 'DS_COR_RACA',
    'instrucao': 'DS_GRAU_INSTRUCAO',
    'ocupacao': 'DS_OCUPACAO',
    'estadoCivil': 'DS_ESTADO_CIVIL',
    'federacao': 'NM_FEDERACAO',
    'siglaFederacao': 'SG_FEDERACAO',
    'ufNascimento': 'SG_UF_NASCIMENTO',
    'tipoAgremiacao': 'TP_AGREMIACAO',
    'composicaoFederacao': 'DS_COMPOSICAO_FEDERACAO',
    'composicaoColigacao': 'DS_COMPOSICAO_COLIGACAO',
    'situacaoCandidatura': 'DS_SITUACAO_CANDIDATURA',
    'dataEleicao': 'DT_ELEICAO',
}

MAPA_COMPLEMENTAR = {
    # Estes campos pertencem ao arquivo complementar.
    'generoFEFC': 'DS_GENERO_FEFC',
    'corRacaFEFC': 'DS_COR_RACA_FEFC',
    'situacaoDetalhada': 'DS_DETALHE_SITUACAO_CAND',
    'statusJulgamento': 'DS_SITUACAO_JULGAMENTO',
    'situacaoPrestacaoContas': 'ST_PREST_CONTAS',
    'situacaoUrna': 'DS_SITUACAO_CANDIDATO_URNA',
    'situacaoTotalizacao': 'DS_SITUACAO_CANDIDATO_TOT',
    'nrProcesso': 'NR_PROCESSO',
    'situacaoCassacao': 'DS_SITUACAO_CASSACAO',
    'situacaoCassacaoMidia': 'DS_SITUACAO_CASSACAO_MIDIA',
    'situacaoDiploma': 'DS_SITUACAO_DIPLOMA',
    'situacaoEleitoral': 'DS_SITUACAO_CANDIDATO_PLEITO',
    'situacaoUrnaInserida': 'ST_CANDIDATO_INSERIDO_URNA',
    'destinacaoVotos': 'NM_TIPO_DESTINACAO_VOTOS',
    'statusJulgamentoPleito': 'DS_SITUACAO_JULGAMENTO_PLEITO',
    'statusJulgamentoUrna': 'DS_SITUACAO_JULGAMENTO_URNA',
    'substituido': 'ST_SUBSTITUIDO',
    'sqSubstituido': 'SQ_SUBSTITUIDO',
    'ordemSuplencia': 'SQ_ORDEM_SUPLENCIA',
    'declarouBens': 'ST_DECLARAR_BENS',
    'protocoloCandidatura': 'NR_PROTOCOLO_CANDIDATURA',
    'aceiteCandidatura': 'DT_ACEITE_CANDIDATURA',
    'municipioNascimento': 'NM_MUNICIPIO_NASCIMENTO',
    'quilombola': 'ST_QUILOMBOLA',
    'etniaIndigena': 'DS_ETNIA_INDIGENA',
    'nacionalidade': 'DS_NACIONALIDADE',
}


def booleano_tse(v):
    t = limpar(v)
    if t is None:
        return None
    if t.upper() in {'S', 'SIM'}:
        return True
    if t.upper() in {'N', 'NAO', 'NÃO'}:
        return False
    return None


def idade_em(nascimento, referencia):
    v = limpar(nascimento)
    if v is None:
        return None
    dt = datetime.strptime(v, '%d/%m/%Y').date()
    if dt > referencia:
        raise ValueError('Nascimento posterior à referência.')
    return (
        referencia.year - dt.year
        - ((referencia.month, referencia.day) < (dt.month, dt.day))
    )


def agrupar(t):
    return {
        nome: {
            k: d for k, d in df.groupby('__chave', sort=False)
        }
        for nome, df in t.items()
        if '__chave' in df
    }


def resumo_monetario(d, fonte_disponivel):
    if not fonte_disponivel:
        return dict(
            total=None, totalCentavos=None, quantidade=None,
            status='sem_fonte', zerosExplicitos=None,
        )
    if d is None or d.empty:
        return dict(
            total=None, totalCentavos=None, quantidade=0,
            status='sem_registros', zerosExplicitos=0,
        )

    vs = d['__centavos'].tolist()
    zeros = sum(v == 0 for v in vs)

    if any(v is None for v in vs):
        return dict(
            total=None, totalCentavos=None, quantidade=len(vs),
            status='valor_nao_informado', zerosExplicitos=zeros,
        )

    n = sum(vs)
    return dict(
        total=reais(n), totalCentavos=str(n), quantidade=len(vs),
        status='ok', zerosExplicitos=zeros,
    )


def registros_publicos(df, cols):
    if df is None:
        return []
    return [
        {
            **{c: limpar(row.get(c)) for c in cols},
            'valorCentavos': (
                None if row.get('__centavos') is None
                else str(row['__centavos'])
            ),
            'fonte': {
                'arquivo': row['__arquivo'],
                'linhaFinalCSV': int(row['__linha']),
            },
        }
        for _, row in df.iterrows()
    ]


def montar_saida(t, base, mapa, cobertura, aud, referencia):
    grupos = agrupar(t)
    cand_grupos = {
        key: d for key, d in t['candidatos'].groupby(CK, sort=False)
    }
    out = {
        n: {} for n in [
            'candidatos', 'patrimonio', 'financeiro', 'juridico',
            'documentos', 'propostas', 'fotos', 'aliases_legados',
        ]
    }
    out['lista_busca'] = []
    aliases = defaultdict(list)
    prestadores_por_candidato = (
        mapa.groupby('__chave')['SQ_PRESTADOR_CONTAS']
        .agg(list).to_dict()
    )

    for _, row in base.iterrows():
        key, uf = row['__chave'], row['SG_UF']
        cp = grupos['complementar'].get(key)
        comp = cp.iloc[0] if cp is not None else {}

        c = dict(
            id=row['SQ_CANDIDATO'],
            chave=key,
            uf=uf,
            anoEleicao=row['ANO_ELEICAO'],
            codigoEleicao=row['CD_ELEICAO'],
        )
        c.update({
            dst: limpar(row.get(src))
            for dst, src in MAPA_PRINCIPAL.items()
        })
        c.update({
            dst: limpar(comp.get(src))
            for dst, src in MAPA_COMPLEMENTAR.items()
        })

        c['situacao'] = c['situacaoDetalhada'] or c['situacaoCandidatura']
        c['municipio'] = (
            c['unidadeEleitoral']
            if (c['abrangencia'] or '').upper() == 'MUNICIPAL'
            else None
        )
        c['tentandoReeleicao'] = booleano_tse(comp.get('ST_REELEICAO'))
        c['reeleicaoValorTSE'] = limpar(comp.get('ST_REELEICAO'))

        try:
            c['idade'] = idade_em(row.get('DT_NASCIMENTO'), referencia)
        except ValueError:
            c['idade'] = None
            aud.registrar(
                'ERRO', 'NASCIMENTO_INVALIDO',
                'Nascimento inválido para cálculo de idade.',
                chave=key,
            )

        c['idadeReferencia'] = referencia.isoformat()
        ip = limpar(comp.get('NR_IDADE_DATA_POSSE'))
        c['idadeDataPosse'] = (
            int(ip) if ip and re.fullmatch(r'\d+', ip) else None
        )
        if ip and c['idadeDataPosse'] is None:
            aud.registrar(
                'ERRO', 'IDADE_POSSE_INVALIDA',
                'Idade na posse inválida.', chave=key,
            )

        c['statusComplementar'] = (
            'ok' if cp is not None
            else 'sem_registros' if uf in cobertura['complementar']
            else 'sem_fonte'
        )

        turnos = cand_grupos[tuple(row[k] for k in CK)]
        c['resultadosPorTurno'] = [
            {
                'turno': limpar(r['NR_TURNO']),
                'resultadoEleitoral': limpar(r['DS_SIT_TOT_TURNO']),
                'codigoResultado': limpar(r['CD_SIT_TOT_TURNO']),
            }
            for _, r in turnos.iterrows()
        ]
        c['turno'] = (
            c['resultadosPorTurno'][0]['turno']
            if len(turnos) == 1 else None
        )
        c['resultadoEleitoral'] = (
            c['resultadosPorTurno'][0]['resultadoEleitoral']
            if len(turnos) == 1 else None
        )
        c['fontes'] = dict(
            candidatura=refs(turnos),
            complementar=[] if cp is None else refs(cp),
        )

        out['candidatos'][key] = c
        aliases[f'{uf}_{c["id"]}'].append(key)

        out['lista_busca'].append({
            **{
                f: c[f] for f in [
                    'id', 'chave', 'uf', 'anoEleicao', 'codigoEleicao',
                    'numeroUrna', 'nomeUrna', 'nomeCompleto',
                    'cargo', 'partido',
                ]
            },
            'nome': c['nomeSocial'] or c['nomeUrna'] or c['nomeCompleto'],
            'nomeBusca': normalizar_busca(
                ' '.join(
                    x or '' for x in [
                        c['nomeCompleto'], c['nomeUrna'], c['nomeSocial']
                    ]
                )
            ),
            'foto': None,
        })

        bens = grupos['bens'].get(key)
        res = resumo_monetario(bens, uf in cobertura['bens'])
        out['patrimonio'][key] = {
            **res,
            'bens': registros_publicos(bens, [
                'NR_ORDEM_BEM_CANDIDATO', 'DS_TIPO_BEM_CANDIDATO',
                'DS_BEM_CANDIDATO', 'VR_BEM_CANDIDATO',
            ]),
        }

        fin = {}
        for familia, total, quant, status in [
            (
                'receitas', 'total_arrecadado',
                'quantidade_receitas', 'status_receitas'
            ),
            (
                'contratadas', 'total_contratado',
                'quantidade_despesas_contratadas', 'status_contratadas'
            ),
            (
                'pagas', 'total_pago',
                'quantidade_despesas_pagas', 'status_pagamento'
            ),
        ]:
            r = resumo_monetario(
                grupos[familia].get(key), uf in cobertura[familia]
            )
            fin.update({
                total: r['total'],
                total + '_centavos': r['totalCentavos'],
                quant: r['quantidade'],
                status: r['status'],
            })
            fin[familia + '_zeros_explicitos'] = r['zerosExplicitos']

        limite = comp.get('__centavos')
        fin['limite_gastos'] = reais(limite)
        fin['limite_gastos_centavos'] = (
            None if limite is None else str(limite)
        )
        fin['percentual_pago'] = (
            float(
                Decimal(fin['total_pago_centavos'])
                / Decimal(fin['total_contratado_centavos']) * 100
            )
            if (
                fin['total_pago_centavos'] is not None
                and fin['total_contratado_centavos'] is not None
                and int(fin['total_contratado_centavos']) > 0
            )
            else None
        )

        fin['prestadores'] = prestadores_por_candidato.get(key, [])
        fin['receitas'] = registros_publicos(
            grupos['receitas'].get(key), [
                'SQ_PRESTADOR_CONTAS', 'SQ_RECEITA', 'DT_RECEITA',
                'DS_RECEITA', 'VR_RECEITA', 'NM_DOADOR',
                'DS_FONTE_RECEITA', 'DS_ORIGEM_RECEITA',
                'DS_NATUREZA_RECEITA', 'DS_ESPECIE_RECEITA',
            ]
        )
        fin['despesasContratadas'] = registros_publicos(
            grupos['contratadas'].get(key), [
                'SQ_PRESTADOR_CONTAS', 'SQ_DESPESA', 'DT_DESPESA',
                'DS_DESPESA', 'VR_DESPESA_CONTRATADA',
                'NM_FORNECEDOR', 'DS_ORIGEM_DESPESA',
            ]
        )
        fin['pagamentos'] = registros_publicos(
            grupos['pagas'].get(key), [
                'SQ_PRESTADOR_CONTAS', 'SQ_DESPESA',
                'SQ_PARCELAMENTO_DESPESA', 'DT_PAGTO_DESPESA',
                'VR_PAGTO_DESPESA', 'DS_FONTE_DESPESA',
            ]
        )
        fin['doadoresOriginarios'] = registros_publicos(
            grupos['originarios'].get(key), [
                'SQ_PRESTADOR_CONTAS', 'SQ_RECEITA',
                'NM_DOADOR_ORIGINARIO', 'TP_DOADOR_ORIGINARIO',
                'VR_RECEITA',
            ]
        )
        fin['statusOriginarios'] = (
            'ok' if grupos['originarios'].get(key) is not None
            else 'sem_registros' if uf in cobertura['originarios']
            else 'sem_fonte'
        )

        detalhes_orig = grupos['originarios'].get(key)
        if (
            detalhes_orig is not None
            and '__originario_repetido' in detalhes_orig.columns
            and detalhes_orig['__originario_repetido'].any()
        ):
            fin['statusOriginarios'] = 'registros_repetidos_na_fonte'

        vers = next(
            (
                grupos[n][key].iloc[0]
                for n in [
                    'receitas', 'contratadas', 'pagas',
                    'receitas_sem_lancamento',
                    'contratadas_sem_lancamento',
                ]
                if key in grupos.get(n, {})
            ),
            {},
        )
        fin['prestacao'] = {
            col: limpar(vers.get(col)) for col in VERSAO_CONTA
        }
        fin['nota'] = (
            'Totais dos lançamentos nas fontes selecionadas. '
            'Receita pode incluir recursos estimáveis; não representa '
            'saldo bancário. Quantidade de despesas pagas = parcelas, '
            'não contratos únicos.'
        )
        out['financeiro'][key] = fin

        cass = grupos['cassacao'].get(key)
        regs = [] if cass is None else [
            {
                **{
                    dest: limpar(r[src])
                    for dest, src in [
                        ('motivo', 'DS_MOTIVO'),
                        ('tipo', 'DS_TP_MOTIVO'),
                        ('processo', 'NR_PROCESSO'),
                    ]
                },
                'fonte': refs(cass.loc[[idx]])[0],
            }
            for idx, r in cass.iterrows()
        ]
        out['juridico'][key] = dict(
            status=(
                'ok' if regs
                else 'sem_registros' if uf in cobertura['cassacao']
                else 'sem_fonte'
            ),
            motivoCassacao=[
                r['motivo'] or r['tipo']
                for r in regs if r['motivo'] or r['tipo']
            ],
            registrosCassacao=regs,
            nota=(
                'Ausência de registro nesta fonte não é certidão negativa '
                'nem conclusão sobre antecedentes ou situação judicial.'
            ),
        )
        out['documentos'][key] = {'propostas': [], 'certidoes': []}
        out['propostas'][key] = []
        out['fotos'][key] = {'arquivo': None, 'status': 'sem_arquivo'}

    out['aliases_legados'] = {
        old: ks[0] for old, ks in aliases.items() if len(ks) == 1
    }
    if len(out['aliases_legados']) != len(aliases):
        aud.registrar(
            'AVISO', 'ALIAS_AMBIGUO',
            'IDs legados que ocorrem em mais de uma eleição '
            'não recebem redirecionamento automático.',
        )

    aud.barreira()
    return out

### Documentos, fotos e propostas com identificação verificável


In [ ]:
def vincular_arquivos(base_dir, base_candidatos, aud, manifest=None):
    """Somente nomes completos ano+UF+SQ ou manifesto explícito com SHA-256."""
    base_dir = Path(base_dir).resolve()
    manifest = manifest or []
    manual = {}
    for item in manifest:
        rel = item['caminhoRelativo']
        if rel in manual:
            aud.registrar('ERRO','MANIFESTO_DUPLICADO','Caminho repetido no manifesto.', arquivo=rel)
        manual[rel] = item
    candidatos = base_candidatos.set_index('__chave')
    validos = set(candidatos.index)
    arquivos, vistos, rejeitados = [], set(), []
    for uf in sorted(base_candidatos['SG_UF'].unique()):
        for pasta, tipo, extensoes in [('Foto_', 'foto', {'.jpg','.jpeg','.png'}),
                                      ('Cert_Criminal_', 'certidao', {'.pdf'}),
                                      ('Proposta_', 'proposta', {'.pdf'})]:
            raiz = base_dir / (pasta + uf)
            if not raiz.is_dir():
                aud.registrar('AVISO','PASTA_DOCUMENTAL_AUSENTE',f'Pasta não disponível: {raiz.name}')
                continue
            for p in sorted(raiz.rglob('*')):
                if not p.is_file():
                    continue
                if p.is_symlink() or not p.resolve().is_relative_to(base_dir):
                    aud.registrar('ERRO','CAMINHO_DOCUMENTAL_INVALIDO','Arquivo fora da pasta do projeto.', arquivo=p.name)
                    continue
                rel = p.relative_to(base_dir).as_posix()
                if p.suffix.lower() not in extensoes:
                    rejeitados.append({'arquivo':rel,'motivo':'extensão não suportada'})
                    continue
                digest = hash_arquivo(p)
                chave, metodo = None, None
                if rel in manual:
                    m = manual[rel]
                    vistos.add(rel)
                    aud.exigir(m.get('sha256') == digest, 'HASH_DOCUMENTO_DIVERGENTE', 'Arquivo não corresponde ao manifesto revisado.', arquivo=rel)
                    aud.exigir(m.get('tipo') == tipo, 'TIPO_DOCUMENTO_DIVERGENTE', 'Categoria do manifesto diverge da pasta.', arquivo=rel)
                    chave = m.get('chave')
                    aud.exigir(chave in validos, 'CANDIDATO_DOCUMENTO_AUSENTE', 'Chave do manifesto não consta na base.', arquivo=rel)
                    if chave in validos:
                        aud.exigir(candidatos.loc[chave,'SG_UF'] == uf, 'UF_DOCUMENTO_DIVERGENTE', 'UF documental diferente da candidatura.', arquivo=rel)
                    metodo = 'manifesto_explicito_sha256'
                else:
                    # Sem substring de CPF/processo: correspondência do nome inteiro.
                    m = re.fullmatch(r'F?(\d{4})([A-Z]{2})(\d{12})(?:[_-][A-Za-z0-9_-]+)?', p.stem)
                    if m:
                        ano, uf_nome, sq = m.groups()
                        hit = base_candidatos[(base_candidatos['ANO_ELEICAO'] == ano) &
                                             (base_candidatos['SG_UF'] == uf_nome) &
                                             (base_candidatos['SQ_CANDIDATO'] == sq)]
                        if len(hit) == 1 and uf == uf_nome:
                            chave = hit.iloc[0]['__chave']
                            metodo = 'nome_completo_ano_uf_sq'
                if chave not in validos:
                    rejeitados.append({'arquivo':rel,'motivo':'identidade ausente/ambígua; exige manifesto'})
                    continue
                arquivos.append(dict(chave=chave, tipo=tipo, origem=str(p), nome=p.name,
                                      sha256=digest, metodoVinculo=metodo, caminhoFonte=rel))
    for rel in set(manual) - vistos:
        aud.registrar('ERRO','MANIFESTO_ARQUIVO_AUSENTE','Item do manifesto não foi localizado em pasta/tipo suportado.', arquivo=rel)
    if rejeitados:
        aud.registrar('AVISO','DOCUMENTOS_NAO_ASSOCIADOS','Arquivos sem vínculo seguro não entram no site.', quantidade=len(rejeitados), arquivos=rejeitados)
    fotos = defaultdict(list)
    for a in arquivos:
        if a['tipo'] == 'foto':
            fotos[a['chave']].append(a)
    ambiguas = {k for k, fs in fotos.items() if len({x['sha256'] for x in fs}) > 1}
    if ambiguas:
        aud.registrar('AVISO','FOTOS_AMBIGUAS','Mais de uma foto diferente para a candidatura; nenhuma escolhida automaticamente.', chaves=sorted(ambiguas))
    # Cópias de bytes idênticos por candidato/tipo não precisam de múltiplos destinos.
    final, usados = [], set()
    for a in arquivos:
        ident = (a['chave'],a['tipo'],a['sha256'])
        if a['tipo'] == 'foto' and a['chave'] in ambiguas:
            continue
        if ident not in usados:
            final.append(a)
            usados.add(ident)
    aud.barreira()
    return final


def extrair_proposta(p):
    try:
        from pypdf import PdfReader
        reader = PdfReader(str(p))
        paginas = [{'pagina': i+1, 'texto': page.extract_text() or ''} for i, page in enumerate(reader.pages)]
        vazias = [x['pagina'] for x in paginas if not x['texto'].strip()]
        status = 'sem_texto_extraivel' if len(vazias) == len(paginas) else 'texto_parcial' if vazias else 'texto_extraido'
        return dict(texto='\n\n'.join(x['texto'] for x in paginas) if status != 'sem_texto_extraivel' else None,
                    paginas=paginas, totalPaginas=len(paginas), paginasSemTexto=vazias,
                    statusExtracao=status, geradoPorIA=False,
                    nota='Extração literal por página, sem OCR. Conferir PDF original; não certifica a leitura integral de imagens/tabelas.')
    except Exception as e:
        return dict(texto=None, paginas=[], totalPaginas=None, paginasSemTexto=None,
                    statusExtracao='erro_extracao', tipoErro=type(e).__name__, geradoPorIA=False)


def copiar_documentos(assets, out, destino, aud):
    for a in assets:
        ext = Path(a['nome']).suffix.lower()
        rel = f'assets/{a["tipo"]}/{a["chave"]}/{a["sha256"]}{ext}'
        p = Path(destino) / rel
        p.parent.mkdir(parents=True, exist_ok=True)
        try:
            if hash_arquivo(a['origem']) != a['sha256']:
                raise ErroIntegridade('Documento mudou após indexação.')
            shutil.copy2(a['origem'], p)
            if hash_arquivo(p) != a['sha256']:
                raise ErroIntegridade('Documento copiado não confere com a fonte.')
        except (OSError, ErroIntegridade) as e:
            aud.registrar('ERRO','COPIA_DOCUMENTO_FALHOU',str(e), arquivo=a['caminhoFonte'])
            continue
        item = dict(nome=a['nome'], caminho=rel, sha256=a['sha256'], metodoVinculo=a['metodoVinculo'])
        key = a['chave']
        if a['tipo'] == 'foto':
            out['fotos'][key] = dict(arquivo=rel,status='ok',sha256=a['sha256'],metodoVinculo=a['metodoVinculo'])
        else:
            categoria = 'propostas' if a['tipo'] == 'proposta' else 'certidoes'
            out['documentos'][key][categoria].append(item)
            if a['tipo'] == 'proposta':
                extracao = extrair_proposta(p)
                out['propostas'][key].append(dict(nome=a['nome'],arquivo=rel,sha256=a['sha256'],**extracao))
                if extracao['statusExtracao'] != 'texto_extraido':
                    aud.registrar('AVISO','PROPOSTA_EXTRACAO_INCOMPLETA','O texto não substitui o PDF; consulte statusExtracao.', chave=key, arquivo=rel)
    for item in out['lista_busca']:
        item['foto'] = out['fotos'][item['chave']]['arquivo']
    aud.barreira()


In [ ]:
import csv
import hashlib
import io
import os
import subprocess
import tempfile
from pathlib import Path

import fitz
from PIL import Image

# Páginas examinadas visualmente nos arquivos enviados.
# A classificação só vale para os mesmos bytes do PDF.
# Não utiliza apenas nome de arquivo ou número do candidato.
PAGINAS_VISUAIS_REVISADAS = {
    # MG130002549557: páginas decorativas, sem texto.
    "740775f9398bd0a22833e9a33bca1e4381e40a492a4083c92daa18fae5798136": {
        2, 100
    },

    # BR280002553884: página de fotografias, sem texto.
    "0159b1f854deec5ec1fe9d28784807353bd61c249680ddaa143b9891b034f33d": {
        27
    },
}


def extrair_proposta(p):
    paginas = []

    try:
        h = hashlib.sha256()
        with open(p, "rb") as arquivo:
            for bloco in iter(
                lambda: arquivo.read(1024 * 1024), b""
            ):
                h.update(bloco)
        digest = h.hexdigest()

        with fitz.open(str(p)) as doc:
            if doc.needs_pass:
                raise ValueError("PDF protegido por senha.")

            total = len(doc)

            for numero, page in enumerate(doc, 1):
                registro = {
                    "pagina": numero,
                    "texto": "",
                    "metodo": None,
                    "status": "pendente",
                    "revisar": False,
                }

                try:
                    nativo = page.get_text(
                        "text", sort=True
                    ).strip()

                    registro["textoNativo"] = nativo

                    # Exceção documentada por revisão visual e hash.
                    if (
                        not nativo
                        and numero in PAGINAS_VISUAIS_REVISADAS.get(
                            digest, set()
                        )
                    ):
                        registro.update(
                            metodo="revisao_visual_sha256",
                            status="pagina_visual_sem_texto",
                        )
                        paginas.append(registro)
                        continue

                    imagens = bool(page.get_image_info())
                    registro["temImagens"] = imagens

                    poucos_caracteres = (
                        sum(c.isalnum() for c in nativo) < 40
                    )
                    suspeito = (
                        "\ufffd" in nativo or "(cid:" in nativo
                    )

                    precisa_ocr = (
                        not nativo
                        or suspeito
                        or (imagens and poucos_caracteres)
                    )

                    if not precisa_ocr:
                        registro.update(
                            texto=nativo,
                            metodo="texto_pdf",
                            status="extraido",
                        )

                    else:
                        # Evita imagens gigantes em páginas com
                        # dimensões internas muito grandes.
                        dpi = max(
                            1,
                            min(
                                300,
                                int(
                                    4000 * 72
                                    / max(
                                        page.rect.width,
                                        page.rect.height,
                                    )
                                ),
                            ),
                        )
                        registro["dpiOCR"] = dpi

                        pix = page.get_pixmap(
                            dpi=dpi,
                            colorspace=fitz.csRGB,
                            alpha=False,
                        )
                        img = Image.frombytes(
                            "RGB",
                            (pix.width, pix.height),
                            pix.samples,
                        )

                        # Não confunde ausência de texto extraível
                        # com página vazia. Confere a renderização.
                        branca = all(
                            minimo >= 250
                            for minimo, maximo in img.getextrema()
                        )

                        if branca and not nativo:
                            registro.update(
                                metodo="renderizacao",
                                status="pagina_branca",
                            )

                        else:
                            with tempfile.TemporaryDirectory() as pasta:
                                png = Path(pasta) / "pagina.png"
                                img.save(png)

                                resultado = subprocess.run(
                                    [
                                        "tesseract",
                                        str(png),
                                        "stdout",
                                        "-l", "por+eng",
                                        "--psm", "3",
                                        "-c", "tessedit_create_tsv=1",
                                    ],
                                    capture_output=True,
                                    text=True,
                                    timeout=120,
                                    check=True,
                                    env={
                                        **os.environ,
                                        "OMP_THREAD_LIMIT": "1",
                                    },
                                )

                            linhas = {}
                            confiancas = []

                            leitor = csv.DictReader(
                                io.StringIO(resultado.stdout),
                                delimiter="\t",
                                quoting=csv.QUOTE_NONE,
                            )

                            for palavra in leitor:
                                txt = (
                                    palavra.get("text") or ""
                                ).strip()

                                if not txt:
                                    continue

                                chave_linha = (
                                    palavra["block_num"],
                                    palavra["par_num"],
                                    palavra["line_num"],
                                )
                                linhas.setdefault(
                                    chave_linha, []
                                ).append(txt)

                                confiancas.append(
                                    float(palavra["conf"])
                                )

                            texto_ocr = "\n".join(
                                " ".join(palavras)
                                for palavras in linhas.values()
                            ).strip()

                            registro["textoOCR"] = texto_ocr

                            # Pontuação do OCR, não probabilidade
                            # de o conteúdo estar correto.
                            registro["confiancaOCRMedia"] = (
                                round(
                                    sum(confiancas) / len(confiancas),
                                    2,
                                )
                                if confiancas else None
                            )

                            registro.update(
                                texto=texto_ocr or nativo,
                                metodo=(
                                    "ocr_tesseract"
                                    if texto_ocr else "texto_pdf"
                                ),
                                status=(
                                    "ocr_revisar"
                                    if texto_ocr
                                    else "sem_texto_revisar"
                                ),
                                revisar=True,
                            )

                    paginas.append(registro)

                except Exception as erro:
                    # Preserva as outras páginas e sinaliza a falha.
                    registro.update(
                        status="erro_pagina",
                        revisar=True,
                        erro=type(erro).__name__,
                        texto=registro.get("textoNativo", ""),
                    )
                    paginas.append(registro)

        pendentes = [
            r["pagina"]
            for r in paginas
            if r["status"] in {
                "erro_pagina", "sem_texto_revisar"
            }
        ]
        revisao = [
            r["pagina"] for r in paginas if r["revisar"]
        ]
        paginas_ocr = [
            r["pagina"]
            for r in paginas
            if r["metodo"] == "ocr_tesseract"
        ]

        texto = "\n\n".join(
            r["texto"] for r in paginas if r["texto"]
        ) or None

        if texto is None:
            status = "sem_texto_extraivel"
        elif pendentes:
            status = "texto_parcial"
        elif paginas_ocr:
            status = "texto_extraido_com_ocr"
        else:
            status = "texto_extraido"

        return {
            "texto": texto,
            "paginas": paginas,
            "totalPaginas": total,
            "sha256Fonte": digest,
            "paginasSemTexto": [
                r["pagina"] for r in paginas if not r["texto"]
            ],
            "paginasBrancas": [
                r["pagina"]
                for r in paginas
                if r["status"] == "pagina_branca"
            ],
            "paginasPendentes": pendentes,
            "paginasOCR": paginas_ocr,
            "paginasParaRevisao": revisao,
            "statusExtracao": status,
            "geradoPorIA": False,
            "aptoParaRascunho": (
                texto is not None and not pendentes
            ),
            "requerRevisaoOCR": bool(revisao),
            "nota": (
                "Texto nativo e OCR preservados por página. "
                "OCR não certifica números, tabelas ou ordem de leitura; "
                "conferir as páginas sinalizadas no PDF. "
                "A extração nativa também pode omitir conteúdo em imagens."
            ),
        }

    except Exception as erro:
        return {
            "texto": None,
            "paginas": paginas,
            "totalPaginas": None,
            "paginasSemTexto": None,
            "paginasPendentes": None,
            "paginasParaRevisao": None,
            "statusExtracao": "erro_extracao",
            "tipoErro": type(erro).__name__,
            "geradoPorIA": False,
            "aptoParaRascunho": False,
            "requerRevisaoOCR": True,
        }

In [ ]:
def vincular_arquivos(base_dir, base_candidatos, aud, manifest=None):
    base_dir = Path(base_dir).resolve()
    manifest = manifest or []

    manual = {}
    for item in manifest:
        rel = item['caminhoRelativo']
        if rel in manual:
            aud.registrar(
                'ERRO', 'MANIFESTO_DUPLICADO',
                'Caminho repetido no manifesto.', arquivo=rel
            )
        manual[rel] = item

    candidatos = base_candidatos.set_index('__chave')
    validos = set(candidatos.index)
    arquivos = []
    vistos = set()
    rejeitados = []

    categorias = [
        ('Foto_', 'foto', {'.jpg', '.jpeg', '.png', '.webp'}),
        ('Cert_Criminal_', 'certidao', {'.pdf'}),
        ('Proposta_', 'proposta', {'.pdf'}),
    ]

    for uf in sorted(base_candidatos['SG_UF'].unique()):
        for pasta, tipo, extensoes in categorias:
            raiz = base_dir / (pasta + uf)

            if not raiz.is_dir():
                aud.registrar(
                    'AVISO', 'PASTA_DOCUMENTAL_AUSENTE',
                    f'Pasta não disponível: {raiz.name}'
                )
                continue

            for p in sorted(raiz.rglob('*')):
                if not p.is_file():
                    continue

                if (
                    p.is_symlink()
                    or not p.resolve().is_relative_to(base_dir)
                ):
                    aud.registrar(
                        'ERRO', 'CAMINHO_DOCUMENTAL_INVALIDO',
                        'Arquivo fora da pasta do projeto.',
                        arquivo=p.name,
                    )
                    continue

                rel = p.relative_to(base_dir).as_posix()

                # Leia-me não é documento de candidato.
                if p.name.casefold() == 'leiame.pdf' and rel not in manual:
                    continue

                if p.suffix.lower() not in extensoes:
                    rejeitados.append({
                        'arquivo': rel,
                        'motivo': 'extensão não suportada',
                    })
                    continue

                digest = hash_arquivo(p)
                chave = None
                metodo = None

                if rel in manual:
                    m = manual[rel]
                    vistos.add(rel)

                    aud.exigir(
                        m.get('sha256') == digest,
                        'HASH_DOCUMENTO_DIVERGENTE',
                        'Arquivo não corresponde ao manifesto revisado.',
                        arquivo=rel,
                    )
                    aud.exigir(
                        m.get('tipo') == tipo,
                        'TIPO_DOCUMENTO_DIVERGENTE',
                        'Categoria do manifesto diverge da pasta.',
                        arquivo=rel,
                    )

                    chave = m.get('chave')
                    aud.exigir(
                        chave in validos,
                        'CANDIDATO_DOCUMENTO_AUSENTE',
                        'Chave do manifesto não consta na base.',
                        arquivo=rel,
                    )

                    if chave in validos:
                        aud.exigir(
                            candidatos.loc[chave, 'SG_UF'] == uf,
                            'UF_DOCUMENTO_DIVERGENTE',
                            'UF documental diferente da candidatura.',
                            arquivo=rel,
                        )

                    metodo = 'manifesto_explicito_sha256'

                else:
                    # Ano + UF + SQ no início.
                    # O sufixo pode conter número de documento, espaços,
                    # acentos e outras extensões.
                    m = re.fullmatch(
                        r'F?([0-9]{4})([A-Z]{2})([0-9]{12})'
                        r'(?:[_-][\s\S]*)?',
                        p.stem,
                    )

                    if m:
                        ano, uf_nome, sq = m.groups()
                        hit = base_candidatos.loc[
                            base_candidatos['ANO_ELEICAO'].eq(ano)
                            & base_candidatos['SG_UF'].eq(uf_nome)
                            & base_candidatos['SQ_CANDIDATO'].eq(sq)
                        ]

                        if len(hit) == 1 and uf == uf_nome:
                            chave = hit.iloc[0]['__chave']
                            metodo = 'nome_completo_ano_uf_sq'

                    elif tipo == 'foto':
                        # Exemplo: FBR280002538811_div.jpg
                        m = re.fullmatch(
                            r'F([A-Z]{2})([0-9]{12})(?:[_-][\s\S]*)?',
                            p.stem,
                        )

                        if m:
                            uf_nome, sq = m.groups()
                            hit = base_candidatos.loc[
                                base_candidatos['SG_UF'].eq(uf_nome)
                                & base_candidatos['SQ_CANDIDATO'].eq(sq)
                            ]

                            # O nome não contém ano/eleição:
                            # só associa quando o resultado é único.
                            if len(hit) == 1 and uf == uf_nome:
                                chave = hit.iloc[0]['__chave']
                                metodo = 'nome_foto_uf_sq_unico_no_escopo'

                if chave not in validos:
                    rejeitados.append({
                        'arquivo': rel,
                        'motivo': 'identidade ausente/ambígua; exige manifesto',
                    })
                    continue

                arquivos.append({
                    'chave': chave,
                    'tipo': tipo,
                    'origem': str(p),
                    'nome': p.name,
                    'sha256': digest,
                    'metodoVinculo': metodo,
                    'caminhoFonte': rel,
                })

    for rel in set(manual) - vistos:
        aud.registrar(
            'ERRO', 'MANIFESTO_ARQUIVO_AUSENTE',
            'Item do manifesto não foi localizado em pasta/tipo suportado.',
            arquivo=rel,
        )

    if rejeitados:
        aud.registrar(
            'AVISO', 'DOCUMENTOS_NAO_ASSOCIADOS',
            'Arquivos sem vínculo seguro não entram no site.',
            quantidade=len(rejeitados),
            arquivos=rejeitados,
        )

    fotos = defaultdict(list)
    for arquivo in arquivos:
        if arquivo['tipo'] == 'foto':
            fotos[arquivo['chave']].append(arquivo)

    ambiguas = {
        chave for chave, itens in fotos.items()
        if len({x['sha256'] for x in itens}) > 1
    }

    if ambiguas:
        aud.registrar(
            'AVISO', 'FOTOS_AMBIGUAS',
            'Mais de uma foto diferente para a candidatura; '
            'nenhuma escolhida automaticamente.',
            chaves=sorted(ambiguas),
        )

    final = []
    usados = set()

    for arquivo in arquivos:
        identidade = (
            arquivo['chave'], arquivo['tipo'], arquivo['sha256']
        )

        if arquivo['tipo'] == 'foto' and arquivo['chave'] in ambiguas:
            continue

        if identidade not in usados:
            final.append(arquivo)
            usados.add(identidade)

    aud.barreira()
    return final

### Auditoria da saída contra as linhas e valores de origem


In [ ]:
def validar_saida(out, t, base, cobertura, aud, referencia):
    """Compara JSON com colunas de origem e soma bruta com Decimal, sem chamar o gerador de totais."""
    keys = set(base['__chave'])
    for nome in ['candidatos','patrimonio','financeiro','juridico','documentos','propostas','fotos']:
        aud.exigir(set(out[nome]) == keys, 'CHAVES_EXPORTACAO', f'{nome}: chaves diferentes da base de candidaturas.')
    aud.exigir(len(out['lista_busca']) == len(keys) and {r['chave'] for r in out['lista_busca']} == keys,
               'BUSCA_DUPLICADA_OU_INCOMPLETA', 'Busca contém chaves duplicadas/ausentes.')
    aud.barreira()
    grupos = agrupar(t)
    turnos_por_candidato = {key: d for key, d in t['candidatos'].groupby(CK, sort=False)}
    for _, row in base.iterrows():
        key = row['__chave']
        c = out['candidatos'][key]
        for dst, src in MAPA_PRINCIPAL.items():
            aud.exigir(c.get(dst) == limpar(row[src]), 'CAMPO_PRINCIPAL_DIVERGENTE', f'{dst} difere da fonte.', chave=key)
        comp_df = grupos['complementar'].get(key)
        comp = comp_df.iloc[0] if comp_df is not None else {}
        for dst, src in MAPA_COMPLEMENTAR.items():
            aud.exigir(c.get(dst) == limpar(comp.get(src)), 'CAMPO_COMPLEMENTAR_DIVERGENTE', f'{dst} difere da fonte.', chave=key)
        for dst, src in [('id','SQ_CANDIDATO'),('codigoEleicao','CD_ELEICAO'),('anoEleicao','ANO_ELEICAO'),('uf','SG_UF')]:
            aud.exigir(c.get(dst) == row[src], 'IDENTIDADE_JSON', f'{dst} errado no JSON.', chave=key)
        raw_turnos = turnos_por_candidato[tuple(row[col] for col in CK)]
        turnos = [{'turno':limpar(r['NR_TURNO']), 'resultadoEleitoral':limpar(r['DS_SIT_TOT_TURNO']),
                   'codigoResultado':limpar(r['CD_SIT_TOT_TURNO'])} for _, r in raw_turnos.iterrows()]
        aud.exigir(c['resultadosPorTurno'] == turnos, 'RESULTADO_TURNO_DIVERGENTE','Resultados não conferem com a base principal.', chave=key)
    busca = {r['chave']:r for r in out['lista_busca']}
    for key, c in out['candidatos'].items():
        for col in ['id','uf','anoEleicao','codigoEleicao','numeroUrna','nomeUrna','nomeCompleto','cargo','partido']:
            aud.exigir(busca[key][col] == c[col], 'BUSCA_DIVERGENTE',f'Busca e ficha divergem em {col}.',chave=key)
    # Reapuração de somas e quantidades a partir das strings originais e da chave eleitoral.
    canonical = {tuple(r[k] for k in CK):r['__chave'] for _,r in base.iterrows()}
    dc = {tuple(r[k] for k in DK):tuple(r[k] for k in CK) for _,r in t['contratadas'].iterrows()}
    checks = [('receitas','financeiro','total_arrecadado','total_arrecadado_centavos','quantidade_receitas','status_receitas','receitas'),
              ('contratadas','financeiro','total_contratado','total_contratado_centavos','quantidade_despesas_contratadas','status_contratadas','despesasContratadas'),
              ('pagas','financeiro','total_pago','total_pago_centavos','quantidade_despesas_pagas','status_pagamento','pagamentos'),
              ('bens','patrimonio','total','totalCentavos','quantidade','status','bens')]
    for nome, target, total, cent, count, status, detalhes in checks:
        sums, counts, missing, origens = defaultdict(int), defaultdict(int), set(), defaultdict(list)
        for _,r in t[nome].iterrows():
            candidate_key = dc[tuple(r[k] for k in DK)] if nome == 'pagas' else tuple(r[k] for k in CK)
            key = canonical[candidate_key]
            counts[key] += 1
            raw = limpar(r[VALORES[nome]])
            if raw is None or raw in {'-1,00','-3,00'}:
                missing.add(key)
                expected_cent = None
            else:
                text = raw.replace('.','').replace(',','.') if aud.escopo.get('formatoMonetario','br') == 'br' else raw
                expected_cent = int(Decimal(text) * 100)
                sums[key] += expected_cent
            origens[key].append((r['__arquivo'],int(r['__linha']),None if expected_cent is None else str(expected_cent)))
        raw_by_ref = {(r['__arquivo'],int(r['__linha'])):r for _,r in t[nome].iterrows()}
        for key in keys:
            uf = out['candidatos'][key]['uf']
            available = uf in cobertura[nome]
            expected_status = 'sem_fonte' if not available else 'sem_registros' if counts[key] == 0 else 'valor_nao_informado' if key in missing else 'ok'
            obj = out[target][key]
            expected_total = sums[key] if expected_status == 'ok' else None
            aud.exigir(obj[count] == (counts[key] if available else None), 'CONTAGEM_DIVERGENTE',f'{nome}: contagem incorreta.',chave=key)
            aud.exigir(obj[status] == expected_status, 'STATUS_DIVERGENTE',f'{nome}: status incorreto.',chave=key)
            aud.exigir(obj[cent] == (str(expected_total) if expected_total is not None else None), 'SOMA_DIVERGENTE',f'{nome}: total não confere com os valores brutos.',chave=key)
            aud.exigir(obj[total] == (float(Decimal(expected_total)/100) if expected_total is not None else None), 'TOTAL_LEGADO_DIVERGENTE',f'{nome}: representação em reais incorreta.',chave=key)
            for registro in obj[detalhes]:
                ref = (registro['fonte']['arquivo'],registro['fonte']['linhaFinalCSV'])
                raw_row = raw_by_ref.get(ref)
                aud.exigir(raw_row is not None, 'FONTE_LANCAMENTO_AUSENTE', 'Linha de origem inexistente no detalhamento.', chave=key)
                if raw_row is not None:
                    for campo, valor in registro.items():
                        if campo not in {'fonte','valorCentavos'}:
                            aud.exigir(campo in raw_row and valor == limpar(raw_row.get(campo)), 'DETALHE_DIVERGENTE', f'{nome}.{campo}: detalhamento diferente da linha de origem.', chave=key)
            got = [(r['fonte']['arquivo'],r['fonte']['linhaFinalCSV'],r['valorCentavos']) for r in obj[detalhes]]
            aud.exigir(sorted(got) == sorted(origens[key]), 'LANCAMENTO_DIVERGENTE',f'{nome}: linhas/valores não pertencem ao candidato.',chave=key)
        aud.reconciliacao[nome] = dict(linhasFonte=len(t[nome]), linhasDetalhadas=sum(len(out[target][k][detalhes]) for k in keys),
                                       valorConhecidoCentavos=str(sum(sums.values())),
                                       candidaturasComValorAusente=len(missing))
    for _, row in base.iterrows():
        key = row['__chave']
        comp_df = grupos['complementar'].get(key)
        comp = comp_df.iloc[0] if comp_df is not None else {}
        c = out['candidatos'][key]
        aud.exigir(c['idade'] == idade_em(row.get('DT_NASCIMENTO'),referencia) and c['idadeReferencia'] == referencia.isoformat(),
                   'IDADE_DIVERGENTE','Idade atual/referência divergente.',chave=key)
        aud.exigir(c['tentandoReeleicao'] == booleano_tse(comp.get('ST_REELEICAO')), 'REELEICAO_DIVERGENTE','Flag de reeleição incorreta.',chave=key)
        f = out['financeiro'][key]
        limite = comp.get('__centavos')
        aud.exigir(f['limite_gastos_centavos'] == (None if limite is None else str(limite)) and f['limite_gastos'] == reais(limite),
                   'LIMITE_DIVERGENTE','Limite não corresponde ao registro complementar.',chave=key)
        cass = grupos['cassacao'].get(key)
        esperado = [] if cass is None else [(limpar(r['DS_MOTIVO']),limpar(r['DS_TP_MOTIVO']),limpar(r['NR_PROCESSO']),r['__arquivo'],int(r['__linha'])) for _,r in cass.iterrows()]
        recebido = [(r['motivo'],r['tipo'],r['processo'],r['fonte']['arquivo'],r['fonte']['linhaFinalCSV']) for r in out['juridico'][key]['registrosCassacao']]
        aud.exigir(recebido == esperado, 'CASSACAO_DIVERGENTE','Motivos/processos diferem da fonte.',chave=key)
        orig = grupos['originarios'].get(key)
        orig_refs = [] if orig is None else [(r['__arquivo'],int(r['__linha'])) for _,r in orig.iterrows()]
        public_orig = f['doadoresOriginarios']
        aud.exigir([(r['fonte']['arquivo'],r['fonte']['linhaFinalCSV']) for r in public_orig] == orig_refs,
                   'ORIGINARIO_DIVERGENTE','Detalhamento originário não corresponde ao candidato/receita.',chave=key)
        if orig is not None and len(orig) == len(public_orig):
            for (_, raw), reg in zip(orig.iterrows(), public_orig):
                for col in ['SQ_PRESTADOR_CONTAS','SQ_RECEITA','NM_DOADOR_ORIGINARIO','TP_DOADOR_ORIGINARIO','VR_RECEITA']:
                    aud.exigir(reg[col] == limpar(raw[col]), 'ORIGINARIO_DIVERGENTE',f'Doador originário diverge em {col}.',chave=key)
                aud.exigir(reg['valorCentavos'] == (None if raw['__centavos'] is None else str(raw['__centavos'])),
                           'ORIGINARIO_DIVERGENTE','Valor originário diverge.',chave=key)
    # JSON estrito: nenhum NaN, Infinity, pandas scalar ou objeto não serializável.
    try:
        json.dumps(out, ensure_ascii=False, allow_nan=False)
    except (ValueError, TypeError) as e:
        aud.registrar('ERRO','JSON_INVALIDO',str(e))
    aud.barreira()


### Exportação isolada, verificação dos bytes e manifesto


In [ ]:
def escrever_json(p, obj):
    Path(p).parent.mkdir(parents=True, exist_ok=True)
    Path(p).write_text(json.dumps(obj,ensure_ascii=False,allow_nan=False,indent=2), encoding='utf-8')


def exportar_validado(out, t, base, cobertura, assets, aud, referencia, pasta_saida):
    # Sempre nova execução. Nenhum dado antigo fica misturado com a exportação atual.
    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)
    run = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
    stage = pasta_saida / ('incompleto_' + run)
    final = pasta_saida / ('validado_' + run)
    stage.mkdir()
    try:
        aud.barreira()
        copiar_documentos(assets, out, stage, aud)
        validar_saida(out, t, base, cobertura, aud, referencia)
        meta = dict(schemaVersion=SCHEMA_VERSION, geradoEm=datetime.now(timezone.utc).isoformat(),
                    fontesOficiais=FONTES_OFICIAIS, arquivosFonte=aud.fontes, escopo=aud.escopo,
                    idadeReferencia=referencia.isoformat(), totalCandidatos=len(base),
                    criterioChave='ANO_ELEICAO_CD_ELEICAO_SG_UF_SQ_CANDIDATO',
                    monetario='*_centavos e totalCentavos são strings de inteiros; null significa não informado. Não usar valor || 0.',
                    status=aud.relatorio()['status'], totalFotos=sum(f['arquivo'] is not None for f in out['fotos'].values()))
        for nome, obj in out.items():
            filename = 'textos_propostas.json' if nome == 'propostas' else nome + '.json'
            escrever_json(stage / 'data' / filename, obj)
        escrever_json(stage/'data'/'metadados.json',meta)
        # Valida os bytes serializados, não só variáveis em memória.
        relido = {nome:json.loads((stage/'data'/('textos_propostas.json' if nome == 'propostas' else nome+'.json')).read_text(encoding='utf-8')) for nome in out}
        validar_saida(relido, t, base, cobertura, aud, referencia)
        for a in assets:
            rel = f'assets/{a["tipo"]}/{a["chave"]}/{a["sha256"]}{Path(a["nome"]).suffix.lower()}'
            aud.exigir(hash_arquivo(stage/rel) == a['sha256'], 'ASSET_EXPORTADO_DIVERGENTE','Asset divergente na validação final.')
        aud.barreira()
        escrever_json(stage/'data'/'verificacao_dados.json',aud.relatorio())
        checksums = {p.relative_to(stage).as_posix():hash_arquivo(p) for p in sorted(stage.rglob('*')) if p.is_file()}
        escrever_json(stage/'manifesto_sha256.json',checksums)
        escrever_json(stage/'EXPORTACAO_VALIDADA.json',dict(status='VALIDADA_TECNICAMENTE',runId=run,
                      manifestoSHA256=hash_arquivo(stage/'manifesto_sha256.json'),
                      nota='Validada contra os arquivos desta execução; dados declaratórios e sujeitos à atualização do TSE.'))
        stage.rename(final)
        return final
    except Exception as e:
        # Mesmo falha de programação ou disco não produz uma pasta com selo de validação.
        aud.registrar('ERRO','EXPORTACAO_INTERROMPIDA',str(e))
        escrever_json(pasta_saida/('auditoria_falha_'+run+'.json'),aud.relatorio())
        raise


def executar_pipeline(base_dir, ano, ufs, catalogo, pasta_saida, referencia=None,
                      escolhas=None, encodings=None, manifest=None, formato='br'):
    aud = Auditoria()
    referencia = referencia or date.today()
    try:
        t, cobertura = carregar_fontes(base_dir, ano, ufs, catalogo, aud, escolhas, encodings)
        aud.escopo['formatoMonetario'] = formato
        t, base, mapa = preparar_bases(t, aud)
        converter_valores(t, aud, formato)
        out = montar_saida(t, base, mapa, cobertura, aud, referencia)
        assets = vincular_arquivos(base_dir, base, aud, manifest)
        for fonte in aud.fontes:
            aud.exigir(hash_arquivo(Path(base_dir)/fonte['arquivo']) == fonte['sha256'], 'FONTE_ALTERADA', 'CSV foi alterado durante o processamento.',arquivo=fonte['arquivo'])
        aud.barreira()
        caminho = exportar_validado(out,t,base,cobertura,assets,aud,referencia,pasta_saida)
        return dict(caminho=caminho, dados=out, auditoria=aud.relatorio())
    except Exception as e:
        aud.registrar('ERRO','EXECUCAO_INTERROMPIDA',str(e))
        destino = Path(pasta_saida)
        destino.mkdir(parents=True,exist_ok=True)
        relatorio = destino/('auditoria_bloqueio_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8]+'.json')
        escrever_json(relatorio,aud.relatorio())
        print('Nenhuma exportação desta execução deve ser publicada. Relatório:',relatorio)
        raise

## 4. Testes de regressão — antes de usar dados reais

Os testes criam arquivos temporários com **pessoas fictícias** e os mesmos cabeçalhos dos anexos. Conferem valores esperados conhecidos, vínculos trocados, colunas ausentes, múltiplas eleições/turnos, contas sem receitas, várias contas por candidato, duplicação, ausências, bytes exportados e bloqueio de saídas adulteradas.

Incluem os dois casos distintos: **1 pagamento com zero explícito** (preservado) e **pagamento com valor ausente** (total `null`). Passar nos testes não valida o conteúdo dos arquivos reais.


In [ ]:
# Testes sintéticos: executam sem Drive e não utilizam pessoas reais.
import copy
import tempfile
import unittest

class TestesIntegridadeTSE(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.catalogo = CATALOGO_COLUNAS

    def fixture(self, root):
        frames = {}
        def row(familia, **kw):
            prefix = FAMILIAS[familia]
            cols = next(cols for name, cols in self.catalogo.items() if name.startswith(prefix+'_2026_'))
            d = dict.fromkeys(cols, '#NULO')
            d.update(DT_GERACAO='11/09/2026', HH_GERACAO='08:00:00')
            d['ANO_ELEICAO' if 'ANO_ELEICAO' in d else 'AA_ELEICAO'] = '2026'
            for k,v in dict(CD_ELEICAO='100',SG_UF='MG',ST_TURNO='1',TP_PRESTACAO_CONTAS='PARCIAL',DT_PRESTACAO_CONTAS='10/09/2026').items():
                if k in d: d[k] = v
            d.update(kw)
            return d
        def pessoa(sq,uf='MG',codigo='100', nome='Pessoa Fictícia'):
            return row('candidatos',SQ_CANDIDATO=sq,SG_UF=uf,CD_ELEICAO=codigo,
                       NR_TURNO='1',CD_CARGO='6',DS_CARGO='DEPUTADO FEDERAL',
                       NR_CANDIDATO='1234',NM_CANDIDATO=nome,NM_URNA_CANDIDATO=nome,
                       NR_CPF_CANDIDATO='00000000001',NM_UE='MINAS GERAIS',SG_UE='MG',
                       TP_ABRANGENCIA='ESTADUAL',SG_PARTIDO='TESTE',DT_NASCIMENTO='20/09/1980',
                       DS_SIT_TOT_TURNO='NÃO ELEITO',CD_SIT_TOT_TURNO='4')
        self.a,self.b,self.c,self.d='000000000001','000000000002','000000000003','000000000004'
        frames['candidatos']=[pessoa(self.a),pessoa(self.b,nome='Outra Pessoa'),pessoa(self.c,nome='Sem Dados'),pessoa(self.d,'BR','200','Pessoa Nacional')]
        frames['complementar']=[row('complementar',SQ_CANDIDATO=self.a,NR_IDADE_DATA_POSSE='46',ST_REELEICAO='N',
                                   ST_SUBSTITUIDO='S',ST_DECLARAR_BENS='S',DS_GENERO_FEFC='FEMININO',
                                   VR_DESPESA_MAX_CAMPANHA='1.234,56',NM_MUNICIPIO_NASCIMENTO='CIDADE TESTE')]
        frames['bens']=[row('bens',SQ_CANDIDATO=self.a,NR_ORDEM_BEM_CANDIDATO='1',VR_BEM_CANDIDATO='25.000,01')]
        frames['cassacao']=[row('cassacao',SQ_CANDIDATO=self.b,NR_PROCESSO='000000000000001',DS_TP_MOTIVO='TIPO TESTE',DS_MOTIVO='MOTIVO FICTÍCIO')]
        frames['receitas']=[row('receitas',SQ_CANDIDATO=self.a,SQ_PRESTADOR_CONTAS='9001',SQ_RECEITA='101',VR_RECEITA='100,10'),
                            row('receitas',SQ_CANDIDATO=self.a,SQ_PRESTADOR_CONTAS='9002',SQ_RECEITA='102',VR_RECEITA='200,20')]
        frames['contratadas']=[row('contratadas',SQ_CANDIDATO=self.a,SQ_PRESTADOR_CONTAS='9001',SQ_DESPESA='301',VR_DESPESA_CONTRATADA='80,00'),
                              row('contratadas',SQ_CANDIDATO=self.a,SQ_PRESTADOR_CONTAS='9002',SQ_DESPESA='302',VR_DESPESA_CONTRATADA='20,00'),
                              row('contratadas',SQ_CANDIDATO=self.b,SQ_PRESTADOR_CONTAS='9003',SQ_DESPESA='303',VR_DESPESA_CONTRATADA='50,00')]
        frames['pagas']=[row('pagas',SQ_PRESTADOR_CONTAS='9001',SQ_DESPESA='301',SQ_PARCELAMENTO_DESPESA='501',VR_PAGTO_DESPESA='30,00'),
                        row('pagas',SQ_PRESTADOR_CONTAS='9001',SQ_DESPESA='301',SQ_PARCELAMENTO_DESPESA='502',VR_PAGTO_DESPESA='50,00'),
                        row('pagas',SQ_PRESTADOR_CONTAS='9002',SQ_DESPESA='302',SQ_PARCELAMENTO_DESPESA='503',VR_PAGTO_DESPESA='20,00'),
                        row('pagas',SQ_PRESTADOR_CONTAS='9003',SQ_DESPESA='303',SQ_PARCELAMENTO_DESPESA='504',VR_PAGTO_DESPESA='0,00')]
        frames['originarios']=[row('originarios',SQ_PRESTADOR_CONTAS='9001',SQ_RECEITA='101',VR_RECEITA='100,10',NM_DOADOR_ORIGINARIO='DOADOR FICTÍCIO')]
        self.write_fixture(root, frames)
        return frames

    def write_fixture(self, root, frames):
        for fam, rows in frames.items():
            # Todo o corpus usa BRASIL para testar o recorte por UF, inclusive complemento sem UF.
            p = Path(root)/f'{FAMILIAS[fam]}_2026_BRASIL.csv'
            cols = next(cols for name,cols in self.catalogo.items() if name.startswith(FAMILIAS[fam]+'_2026_'))
            with p.open('w',encoding='utf-8-sig',newline='') as f:
                w=csv.DictWriter(f,fieldnames=cols,delimiter=';'); w.writeheader(); w.writerows(rows)

    def process(self, root):
        aud=Auditoria()
        t,cob=carregar_fontes(root,2026,['BR','MG'],self.catalogo,aud)
        t,base,mapa=preparar_bases(t,aud)
        converter_valores(t,aud)
        out=montar_saida(t,base,mapa,cob,aud,date(2026,9,11))
        validar_saida(out,t,base,cob,aud,date(2026,9,11))
        return t,base,mapa,cob,out,aud

    def must_block(self, mutate, codigo):
        with tempfile.TemporaryDirectory() as root:
            frames=self.fixture(root); mutate(frames); self.write_fixture(root,frames)
            with self.assertRaises(ErroIntegridade) as caught:
                self.process(root)
            self.assertIn(codigo,str(caught.exception))

    def test_moeda_exata_e_sentinelas(self):
        for entrada, esperado in [('0,00',0),('1.234,56',123456),('1234,56',123456),('1.234',123400),('0,01',1),('#NULO',None),('-1',None),('-3,00',None)]:
            self.assertEqual(centavos(entrada),esperado)
        self.assertEqual(centavos('1234.56','decimal_ponto'),123456)
        self.assertEqual(sum(centavos('0,01') for _ in range(100)),100)
        for entrada in ['1234.56','1,234.56','1,001','abc','NaN1','-10,00','1e3','1..000,00']:
            with self.assertRaises(ValueError): centavos(entrada)

    def test_ids_nao_perdem_digitos(self):
        self.assertEqual(id_texto('000000000001'),'000000000001')
        self.assertEqual(id_texto('12345678901234567890'),'12345678901234567890')
        for entrada in [1.0,123,'123.0','1e12','#NULO','-1']:
            with self.assertRaises(ValueError): id_texto(entrada)

    def test_candidato_e_financeiro_corretos(self):
        with tempfile.TemporaryDirectory() as root:
            self.fixture(root); t,base,mapa,cob,out,aud=self.process(root)
            a='2026_100_MG_'+self.a; b='2026_100_MG_'+self.b; c='2026_100_MG_'+self.c
            self.assertEqual(out['financeiro'][a]['total_pago_centavos'],'10000')
            self.assertEqual(out['financeiro'][a]['quantidade_despesas_pagas'],3)
            self.assertEqual(out['financeiro'][a]['total_arrecadado_centavos'],'30030')
            self.assertEqual(set(out['financeiro'][a]['prestadores']),{'9001','9002'})
            self.assertEqual(out['financeiro'][b]['total_pago_centavos'],'0')
            self.assertEqual(out['financeiro'][b]['quantidade_despesas_pagas'],1)
            self.assertEqual(out['financeiro'][b]['status_pagamento'],'ok')
            self.assertEqual(out['financeiro'][c]['status_pagamento'],'sem_registros')
            self.assertIsNone(out['financeiro'][c]['total_pago'])
            self.assertEqual(out['candidatos'][a]['idade'],45)
            self.assertEqual(out['candidatos'][a]['idadeDataPosse'],46)
            self.assertEqual(out['candidatos'][a]['substituido'],'S')
            self.assertEqual(out['candidatos'][a]['declarouBens'],'S')
            self.assertEqual(out['candidatos'][a]['municipioNascimento'],'CIDADE TESTE')
            self.assertIsNone(out['candidatos'][a]['genero']) # FEFC não substitui dado biográfico.
            self.assertEqual(out['candidatos'][a]['generoFEFC'],'FEMININO')
            self.assertEqual(out['candidatos'][a]['resultadoEleitoral'],'NÃO ELEITO')
            self.assertIsNone(out['candidatos'][a]['municipio'])
            self.assertEqual(out['patrimonio'][a]['totalCentavos'],'2500001')
            self.assertEqual(out['juridico'][b]['registrosCassacao'][0]['processo'],'000000000000001')

    def test_valor_ausente_nao_vira_zero(self):
        with tempfile.TemporaryDirectory() as root:
            frames=self.fixture(root); frames['pagas'][0]['VR_PAGTO_DESPESA']='#NULO';self.write_fixture(root,frames)
            *_,out,aud=self.process(root)
            f=out['financeiro']['2026_100_MG_'+self.a]
            self.assertEqual(f['quantidade_despesas_pagas'],3)
            self.assertIsNone(f['total_pago'])
            self.assertEqual(f['status_pagamento'],'valor_nao_informado')

    def test_coluna_pagamento_errada_bloqueia(self):
        with tempfile.TemporaryDirectory() as root:
            self.fixture(root);p=Path(root)/'despesas_pagas_candidatos_2026_BRASIL.csv'
            p.write_text(p.read_text(encoding='utf-8-sig').replace('VR_PAGTO_DESPESA','VR_DESPESA_PAGA'),encoding='utf-8')
            with self.assertRaises(ErroIntegridade):self.process(root)

    def test_prestador_ambiguo(self):
        self.must_block(lambda x:x['receitas'][0].update(SQ_CANDIDATO=self.b),'PRESTADOR_AMBIGUO')

    def test_pagamento_orfao(self):
        self.must_block(lambda x:x['pagas'][0].update(SQ_PRESTADOR_CONTAS='999999'),'VINCULO_AUSENTE')

    def test_despesa_de_outro_candidato(self):
        self.must_block(lambda x:x['pagas'][0].update(SQ_DESPESA='303'),'VINCULO_AUSENTE')

    def test_outra_eleicao_nao_herda_pagamento(self):
        self.must_block(lambda x:x['pagas'][0].update(CD_ELEICAO='999'),'VINCULO_AUSENTE')

    def test_pagamento_duplicado(self):
        self.must_block(lambda x:x['pagas'].append(copy.deepcopy(x['pagas'][0])),'REGISTRO_REPETIDO_NA_FONTE')

    def test_pagamento_superior_ao_contrato(self):
        self.must_block(lambda x:x['pagas'][0].update(VR_PAGTO_DESPESA='999,00'),'PAGO_SUPERA_CONTRATO')

    def test_mistura_retificacoes(self):
        self.must_block(lambda x:x['pagas'][0].update(DT_PRESTACAO_CONTAS='09/09/2026'),'MULTIPLAS_PRESTACOES')

    def test_identidade_contraditoria(self):
        self.must_block(lambda x:x['receitas'][0].update(NR_CPF_CANDIDATO='00000000099'),'IDENTIDADE_DIVERGENTE')

    def test_valor_invalido_bloqueia(self):
        self.must_block(lambda x:x['pagas'][0].update(VR_PAGTO_DESPESA='INVALIDO'),'MOEDA_INVALIDA')

    def test_bem_duplicado(self):
        self.must_block(lambda x:x['bens'].append(copy.deepcopy(x['bens'][0])),'REGISTRO_REPETIDO_NA_FONTE')

    def test_fonte_ausente_diferente_de_zero(self):
        with tempfile.TemporaryDirectory() as root:
            self.fixture(root);(Path(root)/'despesas_pagas_candidatos_2026_BRASIL.csv').unlink()
            *_,out,aud=self.process(root)
            f=out['financeiro']['2026_100_MG_'+self.a]
            self.assertIsNone(f['total_pago']); self.assertIsNone(f['quantidade_despesas_pagas'])
            self.assertEqual(f['status_pagamento'],'sem_fonte')

    def test_arquivo_nacional_nao_soma_mg_novamente(self):
        with tempfile.TemporaryDirectory() as root:
            self.fixture(root)
            shutil.copy2(Path(root)/'receitas_candidatos_2026_BRASIL.csv',Path(root)/'receitas_candidatos_2026_MG.csv')
            *_,out,aud=self.process(root)
            self.assertEqual(out['financeiro']['2026_100_MG_'+self.a]['total_arrecadado_centavos'],'30030')

    def test_turnos_preservados(self):
        with tempfile.TemporaryDirectory() as root:
            f=self.fixture(root);r=copy.deepcopy(f['candidatos'][0]);r.update(NR_TURNO='2',DS_SIT_TOT_TURNO='ELEITO',CD_SIT_TOT_TURNO='1')
            f['candidatos'].append(r);self.write_fixture(root,f)
            *_,out,aud=self.process(root)
            c=out['candidatos']['2026_100_MG_'+self.a]
            self.assertEqual(len(c['resultadosPorTurno']),2);self.assertIsNone(c['resultadoEleitoral']);self.assertIsNone(c['turno'])

    def test_csv_linha_curta_bloqueia(self):
        with tempfile.TemporaryDirectory() as root:
            self.fixture(root)
            with (Path(root)/'receitas_candidatos_2026_BRASIL.csv').open('a') as f:f.write('2026;INCOMPLETO\n')
            with self.assertRaises(ErroIntegridade):self.process(root)

    def test_csv_encoding_latin1(self):
        with tempfile.TemporaryDirectory() as root:
            self.fixture(root);p=Path(root)/'consulta_cand_2026_BRASIL.csv'
            p.write_bytes(p.read_text(encoding='utf-8-sig').encode('latin1'))
            *_,out,aud=self.process(root)
            self.assertEqual(out['candidatos']['2026_100_MG_'+self.a]['nomeCompleto'],'Pessoa Fictícia')

    def test_json_adulterado_e_detectado(self):
        with tempfile.TemporaryDirectory() as root:
            self.fixture(root); t,base,mapa,cob,out,aud=self.process(root)
            out['financeiro']['2026_100_MG_'+self.a]['total_pago_centavos']='0'
            with self.assertRaises(ErroIntegridade):validar_saida(out,t,base,cob,aud,date(2026,9,11))

    def test_documento_exige_identidade_completa(self):
        with tempfile.TemporaryDirectory() as root:
            self.fixture(root);t,base,mapa,cob,out,aud=self.process(root)
            p=Path(root)/'Proposta_MG';p.mkdir()
            (p/('2026MG'+self.a+'.pdf')).write_bytes(b'%PDF-test')
            (p/('processo123'+self.b+'45.pdf')).write_bytes(b'%PDF-outro')
            (p/('2024MG'+self.a+'.pdf')).write_bytes(b'%PDF-antigo')
            assets=vincular_arquivos(root,base,aud)
            self.assertEqual(len(assets),1);self.assertEqual(assets[0]['chave'],'2026_100_MG_'+self.a)

    def test_fotos_ambiguas_nao_sao_escolhidas(self):
        with tempfile.TemporaryDirectory() as root:
            self.fixture(root);t,base,mapa,cob,out,aud=self.process(root)
            p=Path(root)/'Foto_MG';p.mkdir()
            (p/('2026MG'+self.a+'_1.jpg')).write_bytes(b'fotoA')
            (p/('2026MG'+self.a+'_2.jpg')).write_bytes(b'fotoB')
            self.assertEqual(vincular_arquivos(root,base,aud),[])

    def test_exportacao_completa_e_manifesto(self):
        with tempfile.TemporaryDirectory() as root:
            self.fixture(root);t,base,mapa,cob,out,aud=self.process(root)
            final=exportar_validado(out,t,base,cob,[],aud,date(2026,9,11),Path(root)/'saida')
            self.assertTrue((final/'EXPORTACAO_VALIDADA.json').is_file())
            manifest=json.loads((final/'manifesto_sha256.json').read_text())
            for rel,h in manifest.items():self.assertEqual(hash_arquivo(final/rel),h)
            self.assertEqual(json.loads((final/'data'/'financeiro.json').read_text())['2026_100_MG_'+self.a]['total_pago_centavos'],'10000')

    def test_exportacao_bloqueada_nao_cria_selo(self):
        with tempfile.TemporaryDirectory() as root:
            self.fixture(root);t,base,mapa,cob,out,aud=self.process(root)
            out['financeiro']['2026_100_MG_'+self.a]['total_pago']=0.0
            with self.assertRaises(ErroIntegridade):exportar_validado(out,t,base,cob,[],aud,date(2026,9,11),Path(root)/'saida')
            self.assertFalse(list((Path(root)/'saida').rglob('EXPORTACAO_VALIDADA.json')))


    def test_duas_eleicoes_mesmo_sq_nao_sobrescrevem(self):
        with tempfile.TemporaryDirectory() as root:
            f=self.fixture(root);r=copy.deepcopy(f['candidatos'][0]);r['CD_ELEICAO']='777'
            f['candidatos'].append(r);self.write_fixture(root,f)
            *_,out,aud=self.process(root)
            self.assertIn('2026_777_MG_'+self.a,out['candidatos'])
            self.assertIn('2026_100_MG_'+self.a,out['candidatos'])
            self.assertNotIn('MG_'+self.a,out['aliases_legados'])
            self.assertIsNone(out['financeiro']['2026_777_MG_'+self.a]['total_pago'])

    def test_parcela_mesmo_numero_em_despesas_distintas(self):
        with tempfile.TemporaryDirectory() as root:
            f=self.fixture(root);f['pagas'][2]['SQ_PARCELAMENTO_DESPESA']='501';self.write_fixture(root,f)
            *_,out,aud=self.process(root)
            self.assertEqual(out['financeiro']['2026_100_MG_'+self.a]['total_pago_centavos'],'10000')

    def test_id_float_textual_e_bloqueado(self):
        self.must_block(lambda x:x['receitas'][0].update(SQ_CANDIDATO='1.0'),'ID_INVALIDO')

    def test_data_impossivel_bloqueia(self):
        self.must_block(lambda x:x['pagas'][0].update(DT_PAGTO_DESPESA='31/02/2026'),'DATA_INVALIDA')

    def test_candidato_duplicado_conflitante_bloqueia(self):
        def change(x):
            r=copy.deepcopy(x['candidatos'][0]);r['NM_CANDIDATO']='NOME CONFLITANTE';x['candidatos'].append(r)
        self.must_block(change,'REGISTRO_REPETIDO_NA_FONTE')

    def test_origem_da_receita_nao_vira_receita_nova(self):
        with tempfile.TemporaryDirectory() as root:
            f=self.fixture(root)
            f['receitas'][0]['SQ_CANDIDATO_DOADOR']=self.b
            self.write_fixture(root,f)
            *_,out,aud=self.process(root)
            self.assertEqual(out['financeiro']['2026_100_MG_'+self.a]['total_arrecadado_centavos'],'30030')
            self.assertIsNone(out['financeiro']['2026_100_MG_'+self.b]['total_arrecadado'])

    def test_originario_sem_receita_bloqueia(self):
        self.must_block(lambda x:x['originarios'][0].update(SQ_RECEITA='99999'),'VINCULO_AUSENTE')

    def test_ano_errado_bloqueia_leitura(self):
        with tempfile.TemporaryDirectory() as root:
            f=self.fixture(root);f['pagas'][0]['AA_ELEICAO']='2024';self.write_fixture(root,f)
            with self.assertRaises(ErroIntegridade):self.process(root)

    def test_cabecalho_duplicado_bloqueia(self):
        with tempfile.TemporaryDirectory() as root:
            self.fixture(root);p=Path(root)/'receitas_candidatos_2026_BRASIL.csv'
            p.write_text(p.read_text(encoding='utf-8-sig').replace('DS_RECEITA;VR_RECEITA','VR_RECEITA;VR_RECEITA'),encoding='utf-8')
            with self.assertRaises(ErroIntegridade):self.process(root)

    def test_detalhe_trocado_e_detectado(self):
        with tempfile.TemporaryDirectory() as root:
            self.fixture(root); t,base,mapa,cob,out,aud=self.process(root)
            out['financeiro']['2026_100_MG_'+self.a]['pagamentos'][0]['SQ_DESPESA']='303'
            with self.assertRaises(ErroIntegridade):validar_saida(out,t,base,cob,aud,date(2026,9,11))

    def test_copias_entre_fontes_so_concilia_se_identicas(self):
        aud=Auditoria()
        d=pd.DataFrame([dict(ANO_ELEICAO='2026',CD_ELEICAO='100',SQ_CANDIDATO='1',NM_CANDIDATO='TESTE',__arquivo='A.csv',__linha='2'),
                        dict(ANO_ELEICAO='2026',CD_ELEICAO='100',SQ_CANDIDATO='1',NM_CANDIDATO='TESTE',__arquivo='B.csv',__linha='2')])
        self.assertEqual(len(unicos(d.copy(),CK,aud,'teste')),1)
        d.loc[1,'NM_CANDIDATO']='OUTRO'
        with self.assertRaises(ErroIntegridade):unicos(d,CK,Auditoria(),'teste')

    def test_proposta_mais_de_20_paginas_nao_trunca(self):
        from pypdf import PdfWriter
        with tempfile.TemporaryDirectory() as root:
            w=PdfWriter()
            for _ in range(23):w.add_blank_page(width=100,height=100)
            p=Path(root)/'proposta.pdf'
            with p.open('wb') as f:w.write(f)
            r=extrair_proposta(p)
            self.assertEqual(r['totalPaginas'],23)
            self.assertEqual(len(r['paginas']),23)
            self.assertIsNone(r['texto'])
            self.assertEqual(r['statusExtracao'],'sem_texto_extraivel')

    def test_pipeline_completo_isolado(self):
        with tempfile.TemporaryDirectory() as root:
            self.fixture(root)
            resultado=executar_pipeline(root,2026,['BR','MG'],self.catalogo,Path(root)/'exportados',referencia=date(2026,9,11))
            self.assertEqual(resultado['auditoria']['reconciliacao']['pagas']['linhasFonte'],4)
            self.assertTrue((resultado['caminho']/'EXPORTACAO_VALIDADA.json').is_file())


def executar_testes():
    suite=unittest.defaultTestLoader.loadTestsFromTestCase(TestesIntegridadeTSE)
    result=unittest.TextTestRunner(verbosity=1).run(suite)
    if not result.wasSuccessful():
        raise ErroIntegridade('Testes de regressão falharam. Não executar/exportar dados reais.')
    return {'testes':result.testsRun,'falhas':len(result.failures),'erros':len(result.errors),'dados':'sintéticos'}


In [ ]:
import unicodedata


def normalizar_busca(texto):
    """Normaliza o texto de pesquisa, preservando os nomes originais."""
    return ''.join(
        caractere
        for caractere in unicodedata.normalize(
            'NFD', texto.casefold()
        )
        if unicodedata.category(caractere) != 'Mn'
    )


# Evita manter resultado antigo se os testes falharem.
RESULTADO_TESTES = None

RESULTADO_TESTES = executar_testes()
print(RESULTADO_TESTES)

## 5. Executar a auditoria dos CSVs reais e exportar

A execução é bloqueada, entre outros casos, por fonte malformada, ID inválido, vínculo ausente/ambíguo, identidade contraditória, parcelas conflitantes, versões de prestação misturadas e soma paga maior que a despesa associada. São alertas de integridade/revisão, **não acusações de irregularidade**.

A regra de prestação é conservadora: mais de um `TP_PRESTACAO_CONTAS`, `DT_PRESTACAO_CONTAS` ou `ST_TURNO` por candidatura exige um recorte coerente do TSE. O notebook não escolhe a maior data nem soma “parcial + final + retificadora”. Somente os cabeçalhos foram fornecidos, portanto uma regra automática de seleção de versões não poderia ser comprovada.

Fontes opcionais ausentes e valores não informados podem ser exportados com status explícito e `null`. Uma tabela vazia é diferente de arquivo ausente. Zero só é produzido como total quando existem lançamentos válidos somando zero. A ausência de linhas recebe `sem_registros`, não comprova ausência de movimentação.

Documentos sem identificação inequívoca são relatados e ficam fora da exportação. A extração de propostas percorre todas as páginas, preserva o PDF e sinaliza páginas sem texto; não executa OCR nem resumo por IA. O vínculo pelo nome identifica arquivo/candidatura; não certifica o conteúdo ou validade jurídica do documento.


In [ ]:
import pandas as pd
from pathlib import Path

arquivo = Path(BASE_DIR) / "receitas_candidatos_2026_BRASIL.csv"

for encoding in ("utf-8-sig", "latin1"):
    try:
        amostra = pd.read_csv(
            arquivo,
            sep=";",
            encoding=encoding,
            dtype=str,
            keep_default_na=False,
            usecols=["NR_CPF_CANDIDATO"],
            nrows=120,
        )
        break
    except UnicodeDecodeError:
        if encoding == "latin1":
            raise

valores = amostra["NR_CPF_CANDIDATO"].str.strip()
negativos = valores[valores.str.fullmatch(r"-\d+", na=False)]

print("Códigos negativos encontrados:")
print(negativos.value_counts().to_dict())

In [ ]:
RESULTADO_DADOS_REAIS = None  # Impede reutilizar resultado anterior caso esta tentativa falhe.
if MODO == "dados_reais":
    if RESULTADO_TESTES["falhas"] or RESULTADO_TESTES["erros"]:
        raise ErroIntegridade("Testes falharam.")
    RESULTADO_DADOS_REAIS = executar_pipeline(
        base_dir=BASE_DIR, ano=ANO_ELEICAO, ufs=UFS,
        catalogo=CATALOGO_COLUNAS, pasta_saida=PASTA_SAIDA,
        referencia=DATA_REFERENCIA_IDADE, escolhas=ESCOLHAS_ARQUIVOS,
        encodings=ENCODINGS, manifest=VINCULOS_DOCUMENTOS, formato=FORMATO_MONETARIO,
    )
    print("Exportação validada tecnicamente:", RESULTADO_DADOS_REAIS["caminho"])
    print("Status:", RESULTADO_DADOS_REAIS["auditoria"]["status"])
    print("Candidatos:", len(RESULTADO_DADOS_REAIS["dados"]["candidatos"]))
    print("Leia data/verificacao_dados.json e data/metadados.json antes de atualizar o site.")
else:
    print("Somente testes. Nenhum dado real foi carregado/exportado.")


In [ ]:
import csv
import json
from pathlib import Path

relatorio = Path(
    "/content/drive/MyDrive/Projeto_Eleicao2026/"
    "exportacoes_auditadas/"
    "auditoria_bloqueio_20260912T134833Z_2f7d8349.json"
)

auditoria = json.loads(relatorio.read_text(encoding="utf-8"))


def localizar_erros_monetarios(obj):
    if isinstance(obj, dict):
        if obj.get("codigo") == "MOEDA_INVALIDA":
            yield obj
        else:
            for valor in obj.values():
                yield from localizar_erros_monetarios(valor)
    elif isinstance(obj, list):
        for valor in obj:
            yield from localizar_erros_monetarios(valor)


erros = list(localizar_erros_monetarios(auditoria))

if not erros:
    print("Nenhum MOEDA_INVALIDA encontrado neste relatório.")

for erro in erros:
    print(json.dumps(erro, ensure_ascii=False, indent=2))

    # A mensagem do conversor contém: tabela.COLUNA: explicação.
    campo = erro.get("mensagem", "").split(":", 1)[0]
    coluna = campo.split(".", 1)[1] if "." in campo else None

    if not coluna:
        continue

    por_arquivo = {}
    for exemplo in erro.get("exemplos", []):
        nome = exemplo.get("arquivo")
        linha = exemplo.get("linhaFinalCSV")
        if nome and linha is not None:
            por_arquivo.setdefault(nome, set()).add(int(linha))

    for nome, linhas in por_arquivo.items():
        encontrados = [
            p for p in Path(BASE_DIR).rglob(Path(nome).name)
            if p.is_file()
        ]

        if len(encontrados) != 1:
            print(
                f"{nome}: encontrados {len(encontrados)} arquivos. "
                "Leitura interrompida para evitar escolher a fonte errada."
            )
            continue

        # Latin-1 permite inspecionar os bytes sem descartar caracteres.
        with encontrados[0].open(
            encoding="latin-1", newline=""
        ) as arquivo:
            leitor = csv.DictReader(
                arquivo, delimiter=";", strict=True
            )

            for registro in leitor:
                if leitor.line_num in linhas:
                    print(json.dumps({
                        "arquivo": nome,
                        "linhaFinalCSV": leitor.line_num,
                        "coluna": coluna,
                        "valorBruto": registro.get(coluna),
                        "representacao": repr(registro.get(coluna)),
                    }, ensure_ascii=False, indent=2))

                if leitor.line_num >= max(linhas):
                    break

In [ ]:
import csv
import json
from pathlib import Path
from collections import defaultdict

caminho_relatorio = Path(PASTA_SAIDA) / (
    "auditoria_bloqueio_20260911T204538Z_d36483f5.json"
)

relatorio = json.loads(
    caminho_relatorio.read_text(encoding="utf-8")
)

eventos = [
    e for e in relatorio.get("eventos", [])
    if e.get("codigo") == "REGISTRO_REPETIDO_NA_FONTE"
]

# Até três grupos para manter a saída legível.
eventos = eventos[:3]

alvos = defaultdict(set)

for evento in eventos:
    for referencia in evento.get("exemplos", []):
        alvos[referencia["arquivo"]].add(
            int(referencia["linhaFinalCSV"])
        )

registros = {}

for nome, linhas in alvos.items():
    caminho_csv = Path(BASE_DIR) / nome

    for encoding in ("utf-8-sig", "latin1"):
        try:
            encontrados = {}

            with caminho_csv.open(
                "r", encoding=encoding, newline=""
            ) as f:
                leitor = csv.DictReader(
                    f, delimiter=";", strict=True
                )

                for registro in leitor:
                    linha = leitor.line_num

                    if linha in linhas:
                        encontrados[linha] = registro

                    if linha >= max(linhas):
                        break

            registros[nome] = encontrados
            break

        except UnicodeDecodeError:
            if encoding == "latin1":
                raise

campos_exibidos = [
    "ANO_ELEICAO", "AA_ELEICAO", "CD_ELEICAO", "SG_UF",
    "SQ_CANDIDATO", "SQ_PRESTADOR_CONTAS",
    "SQ_RECEITA", "SQ_DESPESA", "SQ_PARCELAMENTO_DESPESA",
    "NR_ORDEM_BEM_CANDIDATO",
    "TP_PRESTACAO_CONTAS", "DT_PRESTACAO_CONTAS", "ST_TURNO",
    "VR_RECEITA", "VR_DESPESA_CONTRATADA", "VR_PAGTO_DESPESA",
    "VR_BEM_CANDIDATO",
    "DT_RECEITA", "DT_DESPESA", "DT_PAGTO_DESPESA",
    "DS_RECEITA", "DS_DESPESA", "DS_BEM_CANDIDATO",
    "CD_FONTE_RECEITA", "CD_NATUREZA_RECEITA",
    "CD_FONTE_DESPESA", "CD_NATUREZA_DESPESA",
    "CD_ESPECIE_RECURSO",
]

for numero, evento in enumerate(eventos, start=1):
    linhas_brutas = []
    exemplos = []

    for ref in evento.get("exemplos", []):
        nome = ref["arquivo"]
        linha = int(ref["linhaFinalCSV"])
        registro = registros.get(nome, {}).get(linha)

        if registro is None:
            raise RuntimeError(
                f"Referência não localizada: {nome}, linha {linha}"
            )

        linhas_brutas.append(registro)

        exemplos.append({
            "arquivo": nome,
            "linhaFinalCSV": linha,
            **{
                campo: registro[campo]
                for campo in campos_exibidos
                if campo in registro
            },
        })

    colunas = set().union(*(r.keys() for r in linhas_brutas))

    diferentes = sorted(
        coluna for coluna in colunas
        if len({r.get(coluna) for r in linhas_brutas}) > 1
    )

    print(json.dumps({
        "grupo": numero,
        "mensagem": evento["mensagem"],
        "colunasComDiferencasNosExemplos": diferentes,
        "registros": exemplos,
    }, ensure_ascii=False, indent=2))

In [ ]:
import csv
import json
from pathlib import Path

arquivo = Path(BASE_DIR) / "receitas_candidatos_2026_BRASIL.csv"

linhas_alvo = {
    860, 1765, 3148, 4366, 4795,
    11006, 11285, 18656, 26903, 43876,
}

campos = [
    "AA_ELEICAO",
    "CD_ELEICAO",
    "SG_UF",
    "SQ_CANDIDATO",
    "SQ_PRESTADOR_CONTAS",
    "SQ_RECEITA",
    "VR_RECEITA",
    "DS_RECEITA",
    "DT_RECEITA",
    "TP_PRESTACAO_CONTAS",
    "DT_PRESTACAO_CONTAS",
    "ST_TURNO",
]


def ler_exemplos(encoding):
    encontrados = []

    with arquivo.open("r", encoding=encoding, newline="") as f:
        leitor = csv.DictReader(f, delimiter=";", strict=True)

        for registro in leitor:
            linha = leitor.line_num

            if linha in linhas_alvo:
                encontrados.append({
                    "linhaFinalCSV": linha,
                    **{campo: registro.get(campo) for campo in campos},
                })

            if linha >= max(linhas_alvo):
                break

    return encontrados


for encoding in ("utf-8-sig", "latin1"):
    try:
        exemplos = ler_exemplos(encoding)
        break
    except UnicodeDecodeError:
        if encoding == "latin1":
            raise

print(json.dumps(exemplos, ensure_ascii=False, indent=2))

## 6. Busca e consulta no Colab

A busca usa nome completo, urna e nome social. A seleção da ficha usa a **chave eleitoral completa**, evitando juntar homônimos ou candidaturas de eleições diferentes. Os campos exibidos vêm da mesma saída auditada do site.


In [ ]:
def abrir_interface(resultado):
    from IPython.display import display, Image
    import ipywidgets as widgets
    data = resultado["dados"]
    lista = data["lista_busca"]
    texto = widgets.Text(description="Nome:", placeholder="Nome completo, urna ou social")
    uf = widgets.Dropdown(description="UF:", options=["Todas"]+sorted({r["uf"] for r in lista}))
    cargo = widgets.Dropdown(description="Cargo:", options=["Todos"]+sorted({r["cargo"] for r in lista if r["cargo"]}))
    partido = widgets.Dropdown(description="Partido:", options=["Todos"]+sorted({r["partido"] for r in lista if r["partido"]}))
    pesquisar = widgets.Button(description="Pesquisar")
    escolher = widgets.Select(description="Candidato:", options=[], rows=8, layout=widgets.Layout(width="100%"))
    consultar = widgets.Button(description="Consultar ficha")
    resultado_busca, ficha = widgets.Output(), widgets.Output()
    def filtrar(_):
        termos = normalizar_busca(texto.value).split()
        encontrados = [r for r in lista if all(t in r["nomeBusca"] for t in termos)
                       and (uf.value=="Todas" or r["uf"]==uf.value)
                       and (cargo.value=="Todos" or r["cargo"]==cargo.value)
                       and (partido.value=="Todos" or r["partido"]==partido.value)]
        escolher.options = [(f"{r['nome']} | {r['cargo']} | {r['partido']} | {r['chave']}",r["chave"]) for r in encontrados[:300]]
        resultado_busca.clear_output()
        with resultado_busca:
            print(f"{len(encontrados)} resultado(s). Exibindo até 300; refine os filtros se necessário.")
    def mostrar(_):
        ficha.clear_output()
        if escolher.value is None:return
        key=escolher.value
        c=data["candidatos"][key];f=data["financeiro"][key]
        with ficha:
            print(c["nomeCompleto"], "|",key)
            foto=data["fotos"][key]["arquivo"]
            if foto:display(Image(filename=str(resultado["caminho"]/foto),width=140))
            campos={k:v for k,v in c.items() if k not in {"fontes","resultadosPorTurno"}}
            display(pd.DataFrame([{"Campo":k,"Valor":"Não informado" if v is None else v} for k,v in campos.items()]))
            print("RESULTADOS POR TURNO")
            display(pd.DataFrame(c["resultadosPorTurno"]))
            print("FINANCEIRO — fontes selecionadas")
            for label,col,q,status in [("Arrecadado","total_arrecadado_centavos","quantidade_receitas","status_receitas"),
                                      ("Contratado","total_contratado_centavos","quantidade_despesas_contratadas","status_contratadas"),
                                      ("Pago (parcelas)","total_pago_centavos","quantidade_despesas_pagas","status_pagamento")]:
                print(f"{label}: {moeda(f[col])} | Registros: {f[q] if f[q] is not None else 'Não informado'} | {f[status]}")
            print("Limite informado:",moeda(f["limite_gastos_centavos"]))
            print(f["nota"])
            for nome in ["receitas","despesasContratadas","pagamentos","doadoresOriginarios"]:
                print(nome);display(pd.DataFrame(f[nome]))
            print("PATRIMÔNIO:",moeda(data["patrimonio"][key]["totalCentavos"]),data["patrimonio"][key]["status"])
            display(pd.DataFrame(data["patrimonio"][key]["bens"]))
            print("JURÍDICO:",data["juridico"][key]["nota"])
            display(pd.DataFrame(data["juridico"][key]["registrosCassacao"]))
            print("DOCUMENTOS:",json.dumps(data["documentos"][key],ensure_ascii=False,indent=2))
            print("Fontes:",json.dumps(c["fontes"],ensure_ascii=False,indent=2))
    pesquisar.on_click(filtrar)
    consultar.on_click(mostrar)
    display(widgets.VBox([texto,widgets.HBox([uf,cargo,partido]),pesquisar,resultado_busca,escolher,consultar,ficha]))

if RESULTADO_DADOS_REAIS is not None:
    abrir_interface(RESULTADO_DADOS_REAIS)
else:
    print("A interface aparecerá após processar os CSVs reais com sucesso.")


## 7. Contrato de saída para o site — versão 3.0

Todos os JSONs ficam em `validado_.../data/`; fotos e PDFs em `assets/`. Publique o **conjunto completo da mesma execução**, sem juntar arquivos de execuções diferentes.

| Arquivo | Conteúdo |
|---|---|
| `lista_busca.json` | Lista de candidatos, chave completa e caminho da foto validada |
| `candidatos.json` | Dicionário por chave; campos mapeados, resultados por turno e referências |
| `financeiro.json` | Totais, status, prestadores, prestação selecionada, receitas, contratos, parcelas e originários |
| `patrimonio.json` | Total/status, quantidade e bens detalhados com fonte |
| `juridico.json` | Motivos/processos vinculados e ressalva de alcance da fonte |
| `documentos.json`, `propostas.json`, `fotos.json` | Caminhos com hash, vínculos e status da extração |
| `aliases_legados.json` | `UF_SQ` → chave completa somente quando não há ambiguidade |
| `metadados.json` | Versão, escopo, geração da exportação e geração/hash de cada fonte |
| `verificacao_dados.json` | Resultado da auditoria, eventos e conciliação dos totais |

Na raiz: `manifesto_sha256.json` permite conferir os arquivos; `EXPORTACAO_VALIDADA.json` só existe após concluir a validação. Isso comprova consistência técnica, não auditoria externa nem autenticidade criptográfica do TSE.

### Ajustes obrigatórios no frontend

- Usar `chave` no formato `ANO_CD_ELEICAO_UF_SQ_CANDIDATO`. Não usar nome ou número de urna como ID. Aliases antigos são opcionais e podem não existir em caso de colisão.
- `candidatos.json` contém apenas candidaturas; `_meta` foi substituído por `metadados.json`.
- Exibir `null` como **Não informado** e `status=sem_registros` como **Sem registros na fonte consultada**. Nunca `valor || 0`, `Number(null)` ou “sem gastos” como substituição automática.
- Para cálculo exato, usar `*_centavos` e `totalCentavos`: **strings de inteiros**, compatíveis com `BigInt`. Os campos de total em reais são numéricos apenas para compatibilidade/apresentação; não usar ponto flutuante para conciliar contas.
- `quantidade_despesas_pagas` conta **parcelas**. Um contrato pode ter várias parcelas. O detalhamento distingue ambos os identificadores.
- As listas de bens e lançamentos preservam os **nomes de colunas do TSE**, além de `valorCentavos` e `fonte`; adaptar a renderização da estrutura antiga.
- `idade` tem `idadeReferencia`; `idadeDataPosse` é separada. `resultadoEleitoral` e `turno` ficam nulos quando há vários turnos: renderizar `resultadosPorTurno`.
- `situacaoPrestacaoContas`, `substituido`, `declarouBens` e outros campos TSE preservam o valor da fonte; não representam por si só julgamento das contas. `tentandoReeleicao` tem tipo booleano/nulo e valor bruto separado.
- Usar `unidadeEleitoral` no cabeçalho; `municipio` só é preenchido quando a abrangência é municipal. Não misturar os campos biográficos com FEFC.
- Propostas não contêm resumo de IA. Exibir o PDF, o texto extraído e `statusExtracao`; texto pode ser parcial mesmo ao percorrer todas as páginas.

**Exemplo JS de apresentação exata de centavos** (não usar para inferir movimento ausente):

```javascript
function moedaCentavos(valor) {
  if (valor === null || valor === undefined) return "Não informado";
  const n = BigInt(valor);
  return "R$ " + (n / 100n).toLocaleString("pt-BR") + "," +
    (n % 100n).toString().padStart(2, "0");
}
```

### Se a execução bloquear

Abra o relatório `auditoria_bloqueio_...json`. Os eventos indicam tabela, regra e, quando aplicável, arquivo/linha final do registro CSV. Em CSVs com texto multilinha, essa referência é a linha física final do registro, não um índice de planilha.

- `MULTIPLAS_PRESTACOES`: não filtre uma data arbitrária só para passar. Confirme o leiaute/recorte e obtenha os arquivos da mesma entrega de contas; a lista de colunas não permite provar qual versão substitui outra.
- `REGISTRO_REPETIDO_NA_FONTE`: conferir se é duplicação ou granularidade adicional não descrita. Nenhum lançamento financeiro é apagado silenciosamente.
- `VINCULO_AUSENTE` / `PRESTADOR_AMBIGUO`: conferir eleição, contas e presença dos contratos/receitas; nunca preencher o candidato por nome/partido.
- `MOEDA_INVALIDA` / `DATA_INVALIDA`: conferir o valor bruto e o leiaute, sem substituir por zero/data aproximada.
- `PAGO_SUPERA_CONTRATO`: revisar a despesa, parcelas e versões; não rotular candidato como irregular automaticamente.
- Avisos documentais: conferir identidade e, se necessário, adicionar item com hash ao manifesto. Sem vínculo, o arquivo não é associado.

Os checks são deliberadamente conservadores. Se uma granularidade legítima do TSE não couber nas chaves verificadas, será necessário revisar a regra com amostras reais e o leiaute, em vez de desativar o bloqueio. Para execuções grandes, o filtro de UF ocorre durante a leitura, mas dados, índices e JSON ainda usam memória; uma falta de RAM interrompe a execução sem criar uma nova pasta validada.
